# Fase 6 — Evaluación definitiva sobre el conjunto G

En esta fase se ejecutan los enfoques O1 y O2 sobre los mismos 18 videos correspondientes a los 9 participantes del conjunto G.

Los parámetros de EAR, MAR, fusión O2 y persistencia temporal fueron calibrados previamente y permanecen congelados durante toda la evaluación.

La evaluación se realiza contra el Ground Truth final consensuado y adjudicado, sin utilizar información de G para modificar umbrales o parámetros del sistema.

In [ ]:
# ============================================================
# FASE 6 - PASO 1
# CONFIGURACIÓN, VIDEOS G Y GROUND TRUTH FINAL
# ============================================================

from pathlib import Path
import re
import math
import pandas as pd
import numpy as np
import cv2


# ============================================================
# 1. RUTAS DEL PROYECTO
# ============================================================

CARPETA_VIDEOS = Path("../data/raw")
CARPETA_MODELOS = Path("../models")
CARPETA_RESULTADOS = Path("../results")
CARPETA_GT = CARPETA_RESULTADOS / "ground_truth"

RUTA_GT_FINAL = CARPETA_GT / "ground_truth_final.csv"

CARPETA_FASE6 = CARPETA_RESULTADOS / "fase6"
CARPETA_FASE6.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. PARÁMETROS CONGELADOS
#    NO MODIFICAR DURANTE LA EVALUACIÓN DE G
# ============================================================

UMBRAL_EAR = 0.1404
UMBRAL_MAR = 0.1624

TAU_OJO = 0.580015
TAU_BOSTEZO = 0.585053

DURACION_CIERRE_SEG = 0.5
DURACION_BOSTEZO_SEG = 3.0

VENTANA_PERCLOS_SEG = 15.0

PESO_CNN = 0.5
PESO_GEOMETRIA = 0.5


# ============================================================
# 3. IDENTIFICAR LOS 18 VIDEOS DEL CONJUNTO G
# ============================================================

patron_video = re.compile(
    r"^V(\d+)_video(\d+)_(normal|pocaluz|lentes)\.mp4$",
    re.IGNORECASE
)

mapa_condicion = {
    "normal": "normal",
    "pocaluz": "poca_luz",
    "lentes": "lentes"
}

filas_g = []

for ruta in sorted(CARPETA_VIDEOS.glob("*.mp4")):

    m = patron_video.match(ruta.name)

    if m is None:
        continue

    numero_v = int(m.group(1))
    numero_video = int(m.group(2))
    condicion_raw = m.group(3).lower()

    cap = cv2.VideoCapture(str(ruta))

    fps = float(
        cap.get(cv2.CAP_PROP_FPS)
    )

    frames = int(
        cap.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    ancho = int(
        cap.get(cv2.CAP_PROP_FRAME_WIDTH)
    )

    alto = int(
        cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
    )

    cap.release()

    filas_g.append({
        "participante": f"P{numero_v:02d}",
        "voluntario_video": f"V{numero_v}",
        "video_num": numero_video,
        "condicion": mapa_condicion[condicion_raw],
        "video": ruta.stem,
        "archivo": ruta.name,
        "ruta_video": str(ruta),
        "fps": fps,
        "frames": frames,
        "ancho": ancho,
        "alto": alto,
        "duracion_s": frames / fps
    })


df_g = pd.DataFrame(filas_g)

df_g = (
    df_g
    .sort_values(
        ["participante", "video_num"]
    )
    .reset_index(drop=True)
)


# ============================================================
# 4. CARGAR GROUND TRUTH DEFINITIVO
# ============================================================

assert RUTA_GT_FINAL.exists(), (
    f"No se encontró: {RUTA_GT_FINAL}"
)
df_gt = pd.read_csv(
    RUTA_GT_FINAL
)

# ============================================================
# 5. VALIDACIONES
# ============================================================

assert len(df_g) == 18, (
    f"Se esperaban 18 videos y se encontraron {len(df_g)}"
)
assert df_g["participante"].nunique() == 9
assert len(df_gt) == 83, (
    f"Se esperaban 83 eventos GT y se encontraron {len(df_gt)}"
)
assert df_gt["video"].nunique() == 18
assert set(df_gt["etiqueta"].unique()) == {
    "cierre_ocular",
    "bostezo"
}
videos_g = set(df_g["video"])
videos_gt = set(df_gt["video"])
assert videos_g == videos_gt, (
    "Los videos de G y los videos del Ground Truth no coinciden."
)

# ============================================================
# 6. MOSTRAR RESUMEN
# ============================================================

print("=== FASE 6 - INICIALIZACIÓN ===")
print("\nParámetros congelados:")
print(f"EAR: {UMBRAL_EAR}")
print(f"MAR: {UMBRAL_MAR}")
print(f"Tau ojo O2: {TAU_OJO}")
print(f"Tau bostezo O2: {TAU_BOSTEZO}")
print(f"Cierre mínimo: {DURACION_CIERRE_SEG} s")
print(f"Bostezo mínimo: {DURACION_BOSTEZO_SEG} s")
print(f"Ventana PERCLOS: {VENTANA_PERCLOS_SEG} s")
print("\nConjunto G:")
print("Participantes:", df_g["participante"].nunique())
print("Videos:", len(df_g))
print(
    df_g["condicion"]
    .value_counts()
)
print("\nGround Truth:")
print("Eventos:", len(df_gt))
print(
    df_gt["etiqueta"]
    .value_counts()
)
print("\nCarpeta de resultados Fase 6:")
print(CARPETA_FASE6)
print("\nInicialización Fase 6: OK")

=== FASE 6 - INICIALIZACIÓN ===

Parámetros congelados:
EAR: 0.1404
MAR: 0.1624
Tau ojo O2: 0.580015
Tau bostezo O2: 0.585053
Cierre mínimo: 0.5 s
Bostezo mínimo: 3.0 s
Ventana PERCLOS: 15.0 s

Conjunto G:
Participantes: 9
Videos: 18
condicion
normal      9
poca_luz    6
lentes      3
Name: count, dtype: int64

Ground Truth:
Eventos: 83
etiqueta
cierre_ocular    56
bostezo          27
Name: count, dtype: int64

Carpeta de resultados Fase 6:
..\results\fase6

Inicialización Fase 6: OK


In [2]:
import onnxruntime as ort
from pathlib import Path

RUTA_T1 = Path("../models/mobilenetv2_ojos_v2.onnx")
RUTA_T2 = Path("../models/mobilenetv2_yawn_dedup.onnx")

assert RUTA_T1.exists(), f"No existe T1: {RUTA_T1}"
assert RUTA_T2.exists(), f"No existe T2: {RUTA_T2}"

providers = ["CPUExecutionProvider"]

sesion_t1 = ort.InferenceSession(str(RUTA_T1), providers=providers)
sesion_t2 = ort.InferenceSession(str(RUTA_T2), providers=providers)

input_t1 = sesion_t1.get_inputs()[0]
output_t1 = sesion_t1.get_outputs()[0]

input_t2 = sesion_t2.get_inputs()[0]
output_t2 = sesion_t2.get_outputs()[0]

print("=== MODELOS ONNX FINALES ===")

print("\nT1 - Estado ocular")
print("Archivo:", RUTA_T1.name)
print("Provider:", sesion_t1.get_providers())
print("Input:", input_t1.name, input_t1.shape, input_t1.type)
print("Output:", output_t1.name, output_t1.shape, output_t1.type)

print("\nT2 - Bostezo")
print("Archivo:", RUTA_T2.name)
print("Provider:", sesion_t2.get_providers())
print("Input:", input_t2.name, input_t2.shape, input_t2.type)
print("Output:", output_t2.name, output_t2.shape, output_t2.type)

assert sesion_t1.get_providers()[0] == "CPUExecutionProvider"
assert sesion_t2.get_providers()[0] == "CPUExecutionProvider"

print("\nModelos ONNX cargados en CPU: OK")

=== MODELOS ONNX FINALES ===

T1 - Estado ocular
Archivo: mobilenetv2_ojos_v2.onnx
Provider: ['CPUExecutionProvider']
Input: entrada ['batch', 3, 224, 224] tensor(float)
Output: salida ['batch', 1] tensor(float)

T2 - Bostezo
Archivo: mobilenetv2_yawn_dedup.onnx
Provider: ['CPUExecutionProvider']
Input: entrada ['batch', 3, 224, 224] tensor(float)
Output: salida ['batch', 1] tensor(float)

Modelos ONNX cargados en CPU: OK


In [19]:
import mediapipe as mp
from pathlib import Path

# Buscar modelo Face Landmarker
candidatos = list(Path("../models").glob("*face*landmarker*.task"))
assert len(candidatos) > 0, "No se encontró el modelo Face Landmarker .task"

RUTA_FACE = candidatos[0]

BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

opciones_face = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=str(RUTA_FACE)),
    running_mode=VisionRunningMode.IMAGE,
    num_faces=1
)

detector = FaceLandmarker.create_from_options(opciones_face)

print("=== MEDIAPIPE FACE LANDMARKER ===")
print("Modelo:", RUTA_FACE.name)
print("Modo:", "IMAGE")
print("Número máximo de rostros:", 1)
print("\nFace Landmarker cargado: OK")

=== MEDIAPIPE FACE LANDMARKER ===
Modelo: face_landmarker.task
Modo: IMAGE
Número máximo de rostros: 1

Face Landmarker cargado: OK


In [4]:
# ============================================================
# FASE 6 - PASO 4
# EAR, MAR Y REGIONES DE RECORTE
# ============================================================

import numpy as np

EAR_OJO_1 = [33,160,158,133,153,144]
EAR_OJO_2 = [362,385,387,263,373,380]

OJO_1_CROP = [33,7,163,144,145,153,154,155,133,173,157,158,159,160,161,246]
OJO_2_CROP = [362,382,381,380,374,373,390,249,263,466,388,387,386,385,384,398]

BOCA_CROP = [61,146,91,181,84,17,314,405,321,375,291,185,40,39,37,0,267,269,270,409,
             78,95,88,178,87,14,317,402,318,324,308,191,80,81,82,13,312,311,310,415]

def calcular_ear_un_ojo(landmarks, indices, w, h):
    p = [np.array([landmarks[i].x*w, landmarks[i].y*h]) for i in indices]
    vertical = (np.linalg.norm(p[1]-p[5]) + np.linalg.norm(p[2]-p[4])) / 2
    horizontal = np.linalg.norm(p[0]-p[3])
    return vertical / horizontal if horizontal > 0 else np.nan

def calcular_ear(landmarks, w, h):
    ear1 = calcular_ear_un_ojo(landmarks, EAR_OJO_1, w, h)
    ear2 = calcular_ear_un_ojo(landmarks, EAR_OJO_2, w, h)
    return ear1, ear2, (ear1 + ear2) / 2

def calcular_mar(landmarks, w, h):
    def p(i):
        return np.array([landmarks[i].x*w, landmarks[i].y*h])
    horizontal = np.linalg.norm(p(61)-p(291))
    v1 = np.linalg.norm(p(13)-p(14))
    v2 = np.linalg.norm(p(82)-p(312))
    v3 = np.linalg.norm(p(87)-p(317))
    return (v1 + v2 + v3) / (3*horizontal) if horizontal > 0 else np.nan

def recortar_region(frame, landmarks, indices, margen):
    h, w = frame.shape[:2]
    xs = [landmarks[i].x*w for i in indices]
    ys = [landmarks[i].y*h for i in indices]

    x1, x2 = min(xs), max(xs)
    y1, y2 = min(ys), max(ys)

    dx = (x2-x1) * margen
    dy = (y2-y1) * margen

    x1 = max(0, int(np.floor(x1-dx)))
    x2 = min(w, int(np.ceil(x2+dx)))
    y1 = max(0, int(np.floor(y1-dy)))
    y2 = min(h, int(np.ceil(y2+dy)))

    if x2 <= x1 or y2 <= y1:
        return None, None

    return frame[y1:y2, x1:x2].copy(), (x1,y1,x2,y2)

def recortar_regiones(frame, landmarks):
    ojo1, bbox_ojo1 = recortar_region(frame, landmarks, OJO_1_CROP, 0.35)
    ojo2, bbox_ojo2 = recortar_region(frame, landmarks, OJO_2_CROP, 0.35)
    boca, bbox_boca = recortar_region(frame, landmarks, BOCA_CROP, 0.20)
    return ojo1, ojo2, boca, bbox_ojo1, bbox_ojo2, bbox_boca

print("Funciones geométricas y recortes cargados: OK")
print("Margen ojos: 35%")
print("Margen boca: 20%")

Funciones geométricas y recortes cargados: OK
Margen ojos: 35%
Margen boca: 20%


In [23]:
from PIL import Image
import numpy as np
import cv2

MEAN = np.array([0.485,0.456,0.406], dtype=np.float32)
STD = np.array([0.229,0.224,0.225], dtype=np.float32)
CLAHE_T1 = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))

def sigmoid(x):
    x = float(x)
    if x >= 0:
        return 1.0/(1.0+np.exp(-x))
    ex = np.exp(x)
    return ex/(1.0+ex)

def normalizar_imagen(img_rgb):
    pil = Image.fromarray(img_rgb)
    pil = pil.resize((224,224), Image.Resampling.BILINEAR)
    arr = np.asarray(pil, dtype=np.float32)/255.0
    arr = (arr-MEAN)/STD
    arr = np.transpose(arr, (2,0,1))
    return np.expand_dims(arr, axis=0).astype(np.float32)

def preprocesar_t1(recorte_bgr):
    gris = cv2.cvtColor(recorte_bgr, cv2.COLOR_BGR2GRAY)
    gris = cv2.convertScaleAbs(gris, alpha=1.0, beta=30)
    gris = CLAHE_T1.apply(gris)
    rgb = cv2.cvtColor(gris, cv2.COLOR_GRAY2RGB)
    return normalizar_imagen(rgb)

def preprocesar_t2(recorte_bgr):
    rgb = cv2.cvtColor(recorte_bgr, cv2.COLOR_BGR2RGB)
    return normalizar_imagen(rgb)

def inferir_t1_ojo(recorte_bgr):
    x = preprocesar_t1(recorte_bgr)
    logit = sesion_t1.run(
        [output_t1.name],
        {input_t1.name: x}
    )[0].reshape(-1)[0]
    p_abierto = sigmoid(logit)
    p_cerrado = 1.0-p_abierto
    return float(p_cerrado)

def inferir_t1_dos_ojos(ojo1, ojo2):
    p1 = inferir_t1_ojo(ojo1)
    p2 = inferir_t1_ojo(ojo2)
    return p1, p2, (p1+p2)/2.0

def inferir_t2(boca):
    x = preprocesar_t2(boca)
    logit = sesion_t2.run(
        [output_t2.name],
        {input_t2.name: x}
    )[0].reshape(-1)[0]
    return float(sigmoid(logit))

print("=== PREPROCESAMIENTO E INFERENCIA ===")
print("T1: gris + brillo +30 + CLAHE + ImageNet")
print("T1 salida: P(cerrado) = 1 - sigmoid(logit)")
print("T1 dos ojos: promedio de P(cerrado)")
print("T2: RGB + ImageNet")
print("T2 salida: P(bostezo) = sigmoid(logit)")
print("\nFunciones T1/T2 cargadas: OK")

=== PREPROCESAMIENTO E INFERENCIA ===
T1: gris + brillo +30 + CLAHE + ImageNet
T1 salida: P(cerrado) = 1 - sigmoid(logit)
T1 dos ojos: promedio de P(cerrado)
T2: RGB + ImageNet
T2 salida: P(bostezo) = sigmoid(logit)

Funciones T1/T2 cargadas: OK


In [15]:
from collections import deque
import math
import numpy as np

class DetectorPersistencia:
    def __init__(self, fps, duracion_min_seg, nombre_evento):
        self.fps = float(fps)
        self.duracion_min_seg = float(duracion_min_seg)
        self.nombre_evento = nombre_evento
        self.frames_minimos = math.ceil(self.fps*self.duracion_min_seg)
        self.reset()

    def reset(self):
        self.contador = 0
        self.frame_inicio = None
        self.alerta_emitida = False

    def actualizar(self, condicion, frame_idx, rostro_detectado=True):
        if not rostro_detectado or condicion is None:
            self.reset()
            return None

        if condicion:
            if self.contador == 0:
                self.frame_inicio = frame_idx
            self.contador += 1

            if self.contador >= self.frames_minimos and not self.alerta_emitida:
                self.alerta_emitida = True
                return {
                    "evento": self.nombre_evento,
                    "frame_inicio_racha": self.frame_inicio,
                    "frame_alerta": frame_idx,
                    "tiempo_inicio_racha_s": self.frame_inicio/self.fps,
                    "tiempo_alerta_s": frame_idx/self.fps,
                    "frames_persistencia": self.contador,
                    "duracion_min_seg": self.duracion_min_seg
                }
        else:
            self.reset()

        return None


class CalculadorPERCLOS:
    def __init__(self, fps, ventana_seg=15.0):
        self.fps = float(fps)
        self.ventana_seg = float(ventana_seg)
        self.frames_ventana = math.ceil(self.fps*self.ventana_seg)
        self.historial = deque(maxlen=self.frames_ventana)

    def actualizar(self, ear):
        if ear is None or not np.isfinite(ear):
            self.historial.append(None)
        else:
            self.historial.append(1 if ear < UMBRAL_EAR else 0)

        validos = [x for x in self.historial if x is not None]

        if len(validos) == 0:
            return np.nan, 0.0

        perclos = sum(validos)/len(validos)
        cobertura = len(validos)/len(self.historial)
        return float(perclos), float(cobertura)


def crear_estado_t3(fps):
    return {
        "o1_cierre": DetectorPersistencia(fps, DURACION_CIERRE_SEG, "cierre_ocular"),
        "o1_bostezo": DetectorPersistencia(fps, DURACION_BOSTEZO_SEG, "bostezo"),
        "o2_cierre": DetectorPersistencia(fps, DURACION_CIERRE_SEG, "cierre_ocular"),
        "o2_bostezo": DetectorPersistencia(fps, DURACION_BOSTEZO_SEG, "bostezo"),
        "perclos": CalculadorPERCLOS(fps, VENTANA_PERCLOS_SEG)
    }


print("=== LÓGICA TEMPORAL T3 ===")
for fps in [15,23.99,29.99,30.01,60.01]:
    cierre = math.ceil(fps*DURACION_CIERRE_SEG)
    bostezo = math.ceil(fps*DURACION_BOSTEZO_SEG)
    print(f"{fps:5.2f} FPS -> cierre: {cierre:3d} frames | bostezo: {bostezo:3d} frames")

print("\nPersistencia y PERCLOS cargados: OK")

=== LÓGICA TEMPORAL T3 ===
15.00 FPS -> cierre:   8 frames | bostezo:  45 frames
23.99 FPS -> cierre:  12 frames | bostezo:  72 frames
29.99 FPS -> cierre:  15 frames | bostezo:  90 frames
30.01 FPS -> cierre:  16 frames | bostezo:  91 frames
60.01 FPS -> cierre:  31 frames | bostezo: 181 frames

Persistencia y PERCLOS cargados: OK


In [7]:
# ============================================================
# FASE 6 - PASO 7
# DECISIÓN O1 / O2 POR FOTOGRAMA
# ============================================================

def evaluar_frame_t3(estado, frame_idx, rostro_detectado, ear=None, mar=None,
                     p_cerrado=None, p_bostezo=None):

    # Sin rostro: no imputar valores y romper persistencias
    if not rostro_detectado:
        perclos, cobertura = estado["perclos"].actualizar(None)

        estado["o1_cierre"].actualizar(None, frame_idx, False)
        estado["o1_bostezo"].actualizar(None, frame_idx, False)
        estado["o2_cierre"].actualizar(None, frame_idx, False)
        estado["o2_bostezo"].actualizar(None, frame_idx, False)

        return {
            "deteccion_rostro": 0,
            "ear": np.nan,
            "mar": np.nan,
            "perclos": perclos,
            "cobertura_perclos": cobertura,
            "p_cerrado": np.nan,
            "p_bostezo": np.nan,
            "d_ear": np.nan,
            "d_mar": np.nan,
            "score_ojo_o2": np.nan,
            "score_bostezo_o2": np.nan,
            "cond_o1_cierre": np.nan,
            "cond_o1_bostezo": np.nan,
            "cond_o2_cierre": np.nan,
            "cond_o2_bostezo": np.nan,
            "alerta_o1_cierre": None,
            "alerta_o1_bostezo": None,
            "alerta_o2_cierre": None,
            "alerta_o2_bostezo": None
        }

    # Geometría O1
    d_ear = int(ear < UMBRAL_EAR)
    d_mar = int(mar > UMBRAL_MAR)

    # PERCLOS
    perclos, cobertura = estado["perclos"].actualizar(ear)

    # O1
    cond_o1_cierre = d_ear
    cond_o1_bostezo = d_mar

    alerta_o1_cierre = estado["o1_cierre"].actualizar(
        cond_o1_cierre, frame_idx, True
    )
    alerta_o1_bostezo = estado["o1_bostezo"].actualizar(
        cond_o1_bostezo, frame_idx, True
    )

    # O2: fusión 50% CNN + 50% geometría
    score_ojo_o2 = PESO_CNN*p_cerrado + PESO_GEOMETRIA*d_ear
    score_bostezo_o2 = PESO_CNN*p_bostezo + PESO_GEOMETRIA*d_mar

    cond_o2_cierre = int(score_ojo_o2 >= TAU_OJO)
    cond_o2_bostezo = int(score_bostezo_o2 >= TAU_BOSTEZO)

    alerta_o2_cierre = estado["o2_cierre"].actualizar(
        cond_o2_cierre, frame_idx, True
    )
    alerta_o2_bostezo = estado["o2_bostezo"].actualizar(
        cond_o2_bostezo, frame_idx, True
    )

    return {
        "deteccion_rostro": 1,
        "ear": float(ear),
        "mar": float(mar),
        "perclos": perclos,
        "cobertura_perclos": cobertura,
        "p_cerrado": float(p_cerrado),
        "p_bostezo": float(p_bostezo),
        "d_ear": d_ear,
        "d_mar": d_mar,
        "score_ojo_o2": float(score_ojo_o2),
        "score_bostezo_o2": float(score_bostezo_o2),
        "cond_o1_cierre": cond_o1_cierre,
        "cond_o1_bostezo": cond_o1_bostezo,
        "cond_o2_cierre": cond_o2_cierre,
        "cond_o2_bostezo": cond_o2_bostezo,
        "alerta_o1_cierre": alerta_o1_cierre,
        "alerta_o1_bostezo": alerta_o1_bostezo,
        "alerta_o2_cierre": alerta_o2_cierre,
        "alerta_o2_bostezo": alerta_o2_bostezo
    }

print("Función de decisión O1/O2 cargada: OK")
print("O1: EAR/MAR + persistencia")
print("O2: 0.5 CNN + 0.5 geometría + persistencia")
print("PERCLOS: indicador complementario, ventana de 15 s")

Función de decisión O1/O2 cargada: OK
O1: EAR/MAR + persistencia
O2: 0.5 CNN + 0.5 geometría + persistencia
PERCLOS: indicador complementario, ventana de 15 s


In [8]:
# FASE 6 - PASO 8: PRUEBA CONTROLADA T3

estado_test = crear_estado_t3(30.0)

resultado_test = evaluar_frame_t3(
    estado=estado_test,
    frame_idx=0,
    rostro_detectado=True,
    ear=0.2377,
    mar=0.1553,
    p_cerrado=0.2456,
    p_bostezo=0.0389
)

print("=== PRUEBA CONTROLADA T3 ===")
print("EAR:", resultado_test["ear"])
print("MAR:", resultado_test["mar"])
print("D(EAR):", resultado_test["d_ear"])
print("D(MAR):", resultado_test["d_mar"])
print("P(cerrado):", resultado_test["p_cerrado"])
print("P(bostezo):", resultado_test["p_bostezo"])
print("Score ojo O2:", round(resultado_test["score_ojo_o2"],4))
print("Score bostezo O2:", round(resultado_test["score_bostezo_o2"],4))
print("O1 cierre:", resultado_test["cond_o1_cierre"])
print("O1 bostezo:", resultado_test["cond_o1_bostezo"])
print("O2 cierre:", resultado_test["cond_o2_cierre"])
print("O2 bostezo:", resultado_test["cond_o2_bostezo"])
print("PERCLOS:", resultado_test["perclos"])

assert resultado_test["d_ear"] == 0
assert resultado_test["d_mar"] == 0
assert resultado_test["cond_o1_cierre"] == 0
assert resultado_test["cond_o1_bostezo"] == 0
assert resultado_test["cond_o2_cierre"] == 0
assert resultado_test["cond_o2_bostezo"] == 0

print("\nPrueba de decisión T3: OK")

=== PRUEBA CONTROLADA T3 ===
EAR: 0.2377
MAR: 0.1553
D(EAR): 0
D(MAR): 0
P(cerrado): 0.2456
P(bostezo): 0.0389
Score ojo O2: 0.1228
Score bostezo O2: 0.0194
O1 cierre: 0
O1 bostezo: 0
O2 cierre: 0
O2 bostezo: 0
PERCLOS: 0.0

Prueba de decisión T3: OK


In [9]:
# FASE 6 - PASO 9: PROCESAMIENTO COMPLETO DE UN FOTOGRAMA

def procesar_frame_completo(frame, frame_idx, estado):
    h, w = frame.shape[:2]

    # MediaPipe trabaja en RGB
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    rgb = np.ascontiguousarray(rgb)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

    resultado_face = detector.detect(mp_image)

    # --------------------------------------------------------
    # SIN ROSTRO
    # --------------------------------------------------------
    if not resultado_face.face_landmarks:
        resultado = evaluar_frame_t3(
            estado=estado,
            frame_idx=frame_idx,
            rostro_detectado=False
        )
        resultado["p_cerrado_ojo1"] = np.nan
        resultado["p_cerrado_ojo2"] = np.nan
        return resultado

    # --------------------------------------------------------
    # LANDMARKS
    # --------------------------------------------------------
    landmarks = resultado_face.face_landmarks[0]

    ear1, ear2, ear = calcular_ear(landmarks, w, h)
    mar = calcular_mar(landmarks, w, h)

    ojo1, ojo2, boca, _, _, _ = recortar_regiones(
        frame, landmarks
    )

    # Validación defensiva de recortes
    if ojo1 is None or ojo2 is None or boca is None:
        resultado = evaluar_frame_t3(
            estado=estado,
            frame_idx=frame_idx,
            rostro_detectado=False
        )
        resultado["p_cerrado_ojo1"] = np.nan
        resultado["p_cerrado_ojo2"] = np.nan
        return resultado

    # --------------------------------------------------------
    # CNN
    # --------------------------------------------------------
    p_ojo1, p_ojo2, p_cerrado = inferir_t1_dos_ojos(
        ojo1, ojo2
    )
    p_bostezo = inferir_t2(boca)

    # --------------------------------------------------------
    # DECISIÓN O1 / O2
    # --------------------------------------------------------
    resultado = evaluar_frame_t3(
        estado=estado,
        frame_idx=frame_idx,
        rostro_detectado=True,
        ear=ear,
        mar=mar,
        p_cerrado=p_cerrado,
        p_bostezo=p_bostezo
    )

    resultado["ear_ojo1"] = float(ear1)
    resultado["ear_ojo2"] = float(ear2)
    resultado["p_cerrado_ojo1"] = float(p_ojo1)
    resultado["p_cerrado_ojo2"] = float(p_ojo2)

    return resultado


print("Función de procesamiento completo por fotograma: OK")
print("MediaPipe → EAR/MAR → recortes → T1/T2 → O1/O2")

Función de procesamiento completo por fotograma: OK
MediaPipe → EAR/MAR → recortes → T1/T2 → O1/O2


In [10]:
# FASE 6 - PASO 10: PRUEBA CON UN FOTOGRAMA REAL DE G

fila = df_g[df_g["video"] == "V1_video1_normal"].iloc[0]
fps = float(fila["fps"])
frame_idx = int(fila["frames"] // 2)

cap = cv2.VideoCapture(fila["ruta_video"])
cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
ok, frame = cap.read()
cap.release()

assert ok, "No se pudo leer el fotograma."

estado_prueba = crear_estado_t3(fps)
resultado = procesar_frame_completo(frame, frame_idx, estado_prueba)

print("=== PRUEBA REAL DE UN FOTOGRAMA ===")
print("Video:", fila["video"])
print("Frame:", frame_idx)
print("Tiempo:", round(frame_idx/fps, 2), "s")
print("Rostro detectado:", resultado["deteccion_rostro"])

if resultado["deteccion_rostro"] == 1:
    print("EAR:", round(resultado["ear"], 4))
    print("MAR:", round(resultado["mar"], 4))
    print("P(cerrado):", round(resultado["p_cerrado"], 4))
    print("P(bostezo):", round(resultado["p_bostezo"], 4))
    print("Score ojo O2:", round(resultado["score_ojo_o2"], 4))
    print("Score bostezo O2:", round(resultado["score_bostezo_o2"], 4))
    print("O1 cierre:", resultado["cond_o1_cierre"])
    print("O1 bostezo:", resultado["cond_o1_bostezo"])
    print("O2 cierre:", resultado["cond_o2_cierre"])
    print("O2 bostezo:", resultado["cond_o2_bostezo"])
    print("PERCLOS:", round(resultado["perclos"], 4))

    assert 0 <= resultado["p_cerrado"] <= 1
    assert 0 <= resultado["p_bostezo"] <= 1

    print("\nProcesamiento completo del fotograma: OK")
else:
    print("\nMediaPipe no detectó rostro en este fotograma.")

=== PRUEBA REAL DE UN FOTOGRAMA ===
Video: V1_video1_normal
Frame: 1170
Tiempo: 39.02 s
Rostro detectado: 1
EAR: 0.0545
MAR: 0.1547
P(cerrado): 0.3063
P(bostezo): 0.2008
Score ojo O2: 0.6532
Score bostezo O2: 0.1004
O1 cierre: 1
O1 bostezo: 0
O2 cierre: 1
O2 bostezo: 0
PERCLOS: 1.0

Procesamiento completo del fotograma: OK


In [11]:
# FASE 6 - PASO 11: PROCESAMIENTO DE UN VIDEO COMPLETO

import time

def procesar_video_completo(fila_video, guardar=True):
    video = fila_video["video"]
    participante = fila_video["participante"]
    condicion = fila_video["condicion"]
    ruta = fila_video["ruta_video"]
    fps = float(fila_video["fps"])

    cap = cv2.VideoCapture(ruta)
    assert cap.isOpened(), f"No se pudo abrir {ruta}"

    estado = crear_estado_t3(fps)
    filas_frames = []
    filas_alertas = []
    frame_idx = 0

    inicio = time.perf_counter()

    while True:
        ok, frame = cap.read()
        if not ok:
            break

        r = procesar_frame_completo(frame, frame_idx, estado)

        filas_frames.append({
            "participante": participante,
            "video": video,
            "condicion": condicion,
            "frame": frame_idx,
            "tiempo_s": frame_idx/fps,
            "deteccion_rostro": r["deteccion_rostro"],
            "ear": r.get("ear", np.nan),
            "ear_ojo1": r.get("ear_ojo1", np.nan),
            "ear_ojo2": r.get("ear_ojo2", np.nan),
            "mar": r.get("mar", np.nan),
            "perclos": r.get("perclos", np.nan),
            "cobertura_perclos": r.get("cobertura_perclos", np.nan),
            "p_cerrado_ojo1": r.get("p_cerrado_ojo1", np.nan),
            "p_cerrado_ojo2": r.get("p_cerrado_ojo2", np.nan),
            "p_cerrado": r.get("p_cerrado", np.nan),
            "p_bostezo": r.get("p_bostezo", np.nan),
            "d_ear": r.get("d_ear", np.nan),
            "d_mar": r.get("d_mar", np.nan),
            "score_ojo_o2": r.get("score_ojo_o2", np.nan),
            "score_bostezo_o2": r.get("score_bostezo_o2", np.nan),
            "cond_o1_cierre": r.get("cond_o1_cierre", np.nan),
            "cond_o1_bostezo": r.get("cond_o1_bostezo", np.nan),
            "cond_o2_cierre": r.get("cond_o2_cierre", np.nan),
            "cond_o2_bostezo": r.get("cond_o2_bostezo", np.nan)
        })

        alertas = [
            ("O1", r.get("alerta_o1_cierre")),
            ("O1", r.get("alerta_o1_bostezo")),
            ("O2", r.get("alerta_o2_cierre")),
            ("O2", r.get("alerta_o2_bostezo"))
        ]

        for enfoque, alerta in alertas:
            if alerta is not None:
                filas_alertas.append({
                    "participante": participante,
                    "video": video,
                    "condicion": condicion,
                    "enfoque": enfoque,
                    **alerta
                })

        frame_idx += 1

        if frame_idx % 500 == 0:
            print(f"{video}: {frame_idx} frames procesados")

    cap.release()
    tiempo_total = time.perf_counter()-inicio

    df_frames = pd.DataFrame(filas_frames)
    df_alertas = pd.DataFrame(filas_alertas)

    resumen = pd.DataFrame([{
        "participante": participante,
        "video": video,
        "condicion": condicion,
        "fps_video": fps,
        "frames_procesados": frame_idx,
        "rostros_detectados": int(df_frames["deteccion_rostro"].sum()),
        "fallos_rostro": int((df_frames["deteccion_rostro"] == 0).sum()),
        "tasa_fallo_rostro_pct": float((df_frames["deteccion_rostro"] == 0).mean()*100),
        "tiempo_procesamiento_s": tiempo_total,
        "fps_procesamiento": frame_idx/tiempo_total if tiempo_total > 0 else np.nan,
        "latencia_media_ms": (tiempo_total/frame_idx)*1000 if frame_idx > 0 else np.nan,
        "alertas_o1": int((df_alertas["enfoque"] == "O1").sum()) if len(df_alertas) else 0,
        "alertas_o2": int((df_alertas["enfoque"] == "O2").sum()) if len(df_alertas) else 0
    }])

    if guardar:
        df_frames.to_csv(CARPETA_FASE6/f"frames_{video}.csv", index=False)
        df_alertas.to_csv(CARPETA_FASE6/f"alertas_{video}.csv", index=False)
        resumen.to_csv(CARPETA_FASE6/f"resumen_{video}.csv", index=False)

    return df_frames, df_alertas, resumen

print("Función de procesamiento de video completo: OK")
print("Salida: frames + alertas O1/O2 + resumen por video")

Función de procesamiento de video completo: OK
Salida: frames + alertas O1/O2 + resumen por video


In [12]:
# FASE 6 - PASO 12: PRIMER VIDEO COMPLETO

fila_v1 = df_g[df_g["video"] == "V1_video1_normal"].iloc[0]

print("Procesando:", fila_v1["video"])
print("Frames esperados:", fila_v1["frames"])
print("FPS:", round(fila_v1["fps"],2))
print()

df_frames_v1, df_alertas_v1, resumen_v1 = procesar_video_completo(
    fila_v1,
    guardar=True
)

print("\n=== RESUMEN V1 ===")
print(resumen_v1.to_string(index=False))

print("\n=== ALERTAS GENERADAS ===")
if len(df_alertas_v1) == 0:
    print("No se generaron alertas.")
else:
    print(
        df_alertas_v1[
            [
                "enfoque",
                "evento",
                "frame_inicio_racha",
                "frame_alerta",
                "tiempo_alerta_s"
            ]
        ].to_string(index=False)
    )

print("\n=== VALIDACIONES ===")
print("Frames procesados:", len(df_frames_v1))
print("Frames esperados:", int(fila_v1["frames"]))
print("Alertas totales:", len(df_alertas_v1))

assert len(df_frames_v1) == int(fila_v1["frames"])
assert df_frames_v1["frame"].iloc[0] == 0
assert df_frames_v1["frame"].iloc[-1] == int(fila_v1["frames"])-1

print("\nPrimer video completo procesado: OK")

Procesando: V1_video1_normal
Frames esperados: 2340
FPS: 29.99

V1_video1_normal: 500 frames procesados
V1_video1_normal: 1000 frames procesados
V1_video1_normal: 1500 frames procesados
V1_video1_normal: 2000 frames procesados

=== RESUMEN V1 ===
participante            video condicion  fps_video  frames_procesados  rostros_detectados  fallos_rostro  tasa_fallo_rostro_pct  tiempo_procesamiento_s  fps_procesamiento  latencia_media_ms  alertas_o1  alertas_o2
         P01 V1_video1_normal    normal  29.987185               2340                2340              0                    0.0              126.676024           18.47232          54.135053           5           9

=== ALERTAS GENERADAS ===
enfoque        evento  frame_inicio_racha  frame_alerta  tiempo_alerta_s
     O1 cierre_ocular                 797           811        27.044886
     O2 cierre_ocular                 829           843        28.112009
     O1 cierre_ocular                1145          1159        38.649843
     O

In [13]:
# FASE 6 - PASO 13: MATCHING DE ALERTAS CONTRA GROUND TRUTH

def evaluar_alertas_eventos(df_alertas, df_gt_video, enfoque):
    alertas = df_alertas[df_alertas["enfoque"] == enfoque].copy()
    resultados_gt = []
    resultados_alertas = []

    for evento in ["cierre_ocular", "bostezo"]:
        gt_tipo = df_gt_video[
            df_gt_video["etiqueta"] == evento
        ].sort_values("frame_inicio_final").reset_index(drop=True)

        al_tipo = alertas[
            alertas["evento"] == evento
        ].sort_values("frame_alerta").reset_index(drop=True)

        gt_detectados = set()

        for idx_a, alerta in al_tipo.iterrows():
            frame_alerta = int(alerta["frame_alerta"])

            coincidencias = gt_tipo[
                (gt_tipo["frame_inicio_final"] <= frame_alerta) &
                (gt_tipo["frame_fin_final"] >= frame_alerta)
            ]

            if len(coincidencias) == 0:
                resultados_alertas.append({
                    "enfoque": enfoque,
                    "evento": evento,
                    "frame_alerta": frame_alerta,
                    "clasificacion": "FP"
                })
                continue

            idx_gt = coincidencias.index[0]

            if idx_gt not in gt_detectados:
                gt_detectados.add(idx_gt)
                resultados_alertas.append({
                    "enfoque": enfoque,
                    "evento": evento,
                    "frame_alerta": frame_alerta,
                    "clasificacion": "TP"
                })
            else:
                resultados_alertas.append({
                    "enfoque": enfoque,
                    "evento": evento,
                    "frame_alerta": frame_alerta,
                    "clasificacion": "DUPLICADA_IGNORADA"
                })

        for idx_gt, gt in gt_tipo.iterrows():
            detectado = idx_gt in gt_detectados

            resultados_gt.append({
                "enfoque": enfoque,
                "evento": evento,
                "frame_inicio_gt": int(gt["frame_inicio_final"]),
                "frame_fin_gt": int(gt["frame_fin_final"]),
                "resultado": "TP" if detectado else "FN"
            })

    df_res_gt = pd.DataFrame(resultados_gt)
    df_res_alertas = pd.DataFrame(resultados_alertas)

    TP = int((df_res_gt["resultado"] == "TP").sum())
    FN = int((df_res_gt["resultado"] == "FN").sum())
    FP = int((df_res_alertas["clasificacion"] == "FP").sum())

    f1 = 2*TP/(2*TP+FP+FN) if (2*TP+FP+FN) > 0 else 0.0

    return df_res_gt, df_res_alertas, {
        "enfoque": enfoque,
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "F1": f1
    }


gt_v1 = df_gt[df_gt["video"] == "V1_video1_normal"].copy()

gt_o1_v1, alertas_o1_v1, metricas_o1_v1 = evaluar_alertas_eventos(
    df_alertas_v1, gt_v1, "O1"
)

gt_o2_v1, alertas_o2_v1, metricas_o2_v1 = evaluar_alertas_eventos(
    df_alertas_v1, gt_v1, "O2"
)

print("=== RESULTADOS V1 ===")
print(pd.DataFrame([metricas_o1_v1, metricas_o2_v1]).to_string(index=False))

print("\n=== ALERTAS O2 CLASIFICADAS ===")
print(alertas_o2_v1.to_string(index=False))

=== RESULTADOS V1 ===
enfoque  TP  FP  FN  F1
     O1   5   0   0 1.0
     O2   5   0   0 1.0

=== ALERTAS O2 CLASIFICADAS ===
enfoque        evento  frame_alerta      clasificacion
     O2 cierre_ocular           843                 TP
     O2 cierre_ocular          1163                 TP
     O2 cierre_ocular          1193 DUPLICADA_IGNORADA
     O2 cierre_ocular          1224 DUPLICADA_IGNORADA
     O2 cierre_ocular          1245 DUPLICADA_IGNORADA
     O2 cierre_ocular          1540                 TP
     O2 cierre_ocular          1600 DUPLICADA_IGNORADA
     O2       bostezo          1864                 TP
     O2       bostezo          2195                 TP


In [14]:
# FASE 6 - PASO 14: PROCESAR LOS 17 VIDEOS RESTANTES

resumenes_nuevos = []

videos_restantes = df_g[
    df_g["video"] != "V1_video1_normal"
].reset_index(drop=True)

print("Videos pendientes:", len(videos_restantes))
print()

for i, fila in videos_restantes.iterrows():
    video = fila["video"]

    print("="*60)
    print(f"[{i+1}/{len(videos_restantes)}] Procesando: {video}")
    print("="*60)

    df_f, df_a, resumen = procesar_video_completo(
        fila,
        guardar=True
    )

    resumenes_nuevos.append(resumen)

    print(
        f"Finalizado: {video} | "
        f"Frames: {len(df_f)} | "
        f"Alertas: {len(df_a)} | "
        f"FPS proc.: {resumen.iloc[0]['fps_procesamiento']:.2f} | "
        f"Latencia: {resumen.iloc[0]['latencia_media_ms']:.2f} ms"
    )
    print()

print("=== PROCESAMIENTO DE LOS 17 VIDEOS COMPLETADO ===")

Videos pendientes: 17

[1/17] Procesando: V1_video2_pocaluz
V1_video2_pocaluz: 500 frames procesados
V1_video2_pocaluz: 1000 frames procesados
Finalizado: V1_video2_pocaluz | Frames: 1477 | Alertas: 14 | FPS proc.: 20.06 | Latencia: 49.84 ms

[2/17] Procesando: V2_video1_normal
V2_video1_normal: 500 frames procesados
V2_video1_normal: 1000 frames procesados
V2_video1_normal: 1500 frames procesados
V2_video1_normal: 2000 frames procesados
Finalizado: V2_video1_normal | Frames: 2100 | Alertas: 13 | FPS proc.: 18.02 | Latencia: 55.49 ms

[3/17] Procesando: V2_video2_lentes
V2_video2_lentes: 500 frames procesados
V2_video2_lentes: 1000 frames procesados
V2_video2_lentes: 1500 frames procesados
Finalizado: V2_video2_lentes | Frames: 1972 | Alertas: 25 | FPS proc.: 19.11 | Latencia: 52.32 ms

[4/17] Procesando: V3_video1_normal
V3_video1_normal: 500 frames procesados
V3_video1_normal: 1000 frames procesados
V3_video1_normal: 1500 frames procesados
V3_video1_normal: 2000 frames procesados
V3_

In [15]:
# FASE 6 - PASO 15: CONSOLIDAR Y VALIDAR LOS 18 VIDEOS

from pathlib import Path
import pandas as pd
import numpy as np

resumenes = []
errores = []

for _, fila in df_g.iterrows():
    video = fila["video"]
    ruta_frames = CARPETA_FASE6/f"frames_{video}.csv"
    ruta_alertas = CARPETA_FASE6/f"alertas_{video}.csv"
    ruta_resumen = CARPETA_FASE6/f"resumen_{video}.csv"

    if not ruta_frames.exists():
        errores.append(f"Falta frames: {video}")
        continue
    if not ruta_alertas.exists():
        errores.append(f"Falta alertas: {video}")
        continue
    if not ruta_resumen.exists():
        errores.append(f"Falta resumen: {video}")
        continue

    r = pd.read_csv(ruta_resumen)
    resumenes.append(r)

df_resumen_fase6 = pd.concat(resumenes, ignore_index=True)

# Validar frames reales vs esperados
esperados = dict(zip(df_g["video"], df_g["frames"]))
df_resumen_fase6["frames_esperados"] = df_resumen_fase6["video"].map(esperados)
df_resumen_fase6["frames_ok"] = (
    df_resumen_fase6["frames_procesados"] ==
    df_resumen_fase6["frames_esperados"]
)

print("=== VALIDACIÓN DE LOS 18 VIDEOS ===")
print("Resúmenes encontrados:", len(df_resumen_fase6))
print("Errores de archivos:", len(errores))
print("Videos con frames completos:", int(df_resumen_fase6["frames_ok"].sum()), "/ 18")

if errores:
    print("\nProblemas:")
    for e in errores:
        print("-", e)

print("\n=== RESUMEN DE PROCESAMIENTO ===")
print(
    df_resumen_fase6[
        ["video","condicion","frames_procesados","fallos_rostro",
         "tasa_fallo_rostro_pct","fps_procesamiento",
         "latencia_media_ms","alertas_o1","alertas_o2"]
    ].to_string(
        index=False,
        formatters={
            "tasa_fallo_rostro_pct": lambda x: f"{x:.2f}",
            "fps_procesamiento": lambda x: f"{x:.2f}",
            "latencia_media_ms": lambda x: f"{x:.2f}"
        }
    )
)

print("\n=== PROMEDIOS PRELIMINARES ===")
print("FPS procesamiento:", round(df_resumen_fase6["fps_procesamiento"].mean(),2))
print("Latencia media:", round(df_resumen_fase6["latencia_media_ms"].mean(),2), "ms")
print("Tasa fallo rostro:", round(
    100*df_resumen_fase6["fallos_rostro"].sum() /
    df_resumen_fase6["frames_procesados"].sum(), 2
), "%")

assert len(df_resumen_fase6) == 18
assert len(errores) == 0
assert df_resumen_fase6["frames_ok"].all()

df_resumen_fase6.to_csv(
    CARPETA_FASE6/"resumen_18_videos.csv",
    index=False
)

print("\nValidación de procesamiento: OK")

=== VALIDACIÓN DE LOS 18 VIDEOS ===
Resúmenes encontrados: 18
Errores de archivos: 0
Videos con frames completos: 18 / 18

=== RESUMEN DE PROCESAMIENTO ===
            video condicion  frames_procesados  fallos_rostro tasa_fallo_rostro_pct fps_procesamiento latencia_media_ms  alertas_o1  alertas_o2
 V1_video1_normal    normal               2340              0                  0.00             18.47             54.14           5           9
V1_video2_pocaluz  poca_luz               1477              0                  0.00             20.06             49.84           7           7
 V2_video1_normal    normal               2100              0                  0.00             18.02             55.49           7           6
 V2_video2_lentes    lentes               1972              0                  0.00             19.11             52.32          10          15
 V3_video1_normal    normal               3734              0                  0.00             18.54             53.93     

In [17]:
def evaluar_alertas_eventos(df_alertas, df_gt_video, enfoque):
    resultados_gt = []
    resultados_alertas = []

    if len(df_alertas) == 0 or "enfoque" not in df_alertas.columns:
        alertas = pd.DataFrame(columns=["enfoque","evento","frame_alerta"])
    else:
        alertas = df_alertas[df_alertas["enfoque"] == enfoque].copy()

    for evento in ["cierre_ocular","bostezo"]:
        gt_tipo = df_gt_video[
            df_gt_video["etiqueta"] == evento
        ].sort_values("frame_inicio_final").reset_index(drop=True)

        if len(alertas) == 0:
            al_tipo = pd.DataFrame(columns=["evento","frame_alerta"])
        else:
            al_tipo = alertas[
                alertas["evento"] == evento
            ].sort_values("frame_alerta").reset_index(drop=True)

        gt_detectados = set()

        for _, alerta in al_tipo.iterrows():
            frame_alerta = int(alerta["frame_alerta"])

            coincidencias = gt_tipo[
                (gt_tipo["frame_inicio_final"] <= frame_alerta) &
                (gt_tipo["frame_fin_final"] >= frame_alerta)
            ]

            if len(coincidencias) == 0:
                resultados_alertas.append({
                    "enfoque": enfoque,
                    "evento": evento,
                    "frame_alerta": frame_alerta,
                    "clasificacion": "FP"
                })
                continue

            idx_gt = coincidencias.index[0]

            if idx_gt not in gt_detectados:
                gt_detectados.add(idx_gt)
                clasificacion = "TP"
            else:
                clasificacion = "DUPLICADA_IGNORADA"

            resultados_alertas.append({
                "enfoque": enfoque,
                "evento": evento,
                "frame_alerta": frame_alerta,
                "clasificacion": clasificacion
            })

        for idx_gt, gt in gt_tipo.iterrows():
            resultados_gt.append({
                "enfoque": enfoque,
                "evento": evento,
                "frame_inicio_gt": int(gt["frame_inicio_final"]),
                "frame_fin_gt": int(gt["frame_fin_final"]),
                "resultado": "TP" if idx_gt in gt_detectados else "FN"
            })

    df_res_gt = pd.DataFrame(
        resultados_gt,
        columns=[
            "enfoque","evento","frame_inicio_gt",
            "frame_fin_gt","resultado"
        ]
    )

    df_res_alertas = pd.DataFrame(
        resultados_alertas,
        columns=[
            "enfoque","evento","frame_alerta","clasificacion"
        ]
    )

    TP = int((df_res_gt["resultado"] == "TP").sum())
    FN = int((df_res_gt["resultado"] == "FN").sum())
    FP = int((df_res_alertas["clasificacion"] == "FP").sum())

    denominador = 2*TP + FP + FN
    f1 = 2*TP/denominador if denominador > 0 else 0.0

    return df_res_gt, df_res_alertas, {
        "enfoque": enfoque,
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "F1": f1
    }

print("Función de evaluación corregida: OK")

Función de evaluación corregida: OK


In [18]:
# FASE 6 - PASO 16: EVALUAR O1 Y O2 EN LOS 18 VIDEOS

metricas_videos = []
detalle_gt_total = []
detalle_alertas_total = []

for _, fila in df_g.iterrows():
    video = fila["video"]
    participante = fila["participante"]
    condicion = fila["condicion"]

    ruta_alertas = CARPETA_FASE6/f"alertas_{video}.csv"

    try:
        df_alertas_video = pd.read_csv(ruta_alertas)
    except pd.errors.EmptyDataError:
        df_alertas_video = pd.DataFrame(
            columns=["enfoque","evento","frame_alerta"]
        )

    gt_video = df_gt[df_gt["video"] == video].copy()

    for enfoque in ["O1","O2"]:
        res_gt, res_alertas, metricas = evaluar_alertas_eventos(
            df_alertas_video,
            gt_video,
            enfoque
        )

        metricas_videos.append({
            "participante": participante,
            "video": video,
            "condicion": condicion,
            **metricas
        })

        if len(res_gt):
            res_gt["participante"] = participante
            res_gt["video"] = video
            res_gt["condicion"] = condicion
            detalle_gt_total.append(res_gt)

        if len(res_alertas):
            res_alertas["participante"] = participante
            res_alertas["video"] = video
            res_alertas["condicion"] = condicion
            detalle_alertas_total.append(res_alertas)

df_metricas_video = pd.DataFrame(metricas_videos)

df_detalle_gt = pd.concat(
    detalle_gt_total,
    ignore_index=True
)

df_detalle_alertas = pd.concat(
    detalle_alertas_total,
    ignore_index=True
) if detalle_alertas_total else pd.DataFrame()

# Guardar
df_metricas_video.to_csv(
    CARPETA_FASE6/"metricas_por_video.csv",
    index=False
)

df_detalle_gt.to_csv(
    CARPETA_FASE6/"detalle_ground_truth_evaluado.csv",
    index=False
)

df_detalle_alertas.to_csv(
    CARPETA_FASE6/"detalle_alertas_clasificadas.csv",
    index=False
)

print("=== MÉTRICAS POR VIDEO ===")
print(
    df_metricas_video[
        ["participante","video","condicion","enfoque","TP","FP","FN","F1"]
    ].to_string(
        index=False,
        formatters={"F1": lambda x: f"{x:.4f}"}
    )
)

print("\n=== TOTALES PRELIMINARES ===")
totales = (
    df_metricas_video
    .groupby("enfoque")[["TP","FP","FN"]]
    .sum()
)

totales["F1_micro"] = (
    2*totales["TP"] /
    (2*totales["TP"] + totales["FP"] + totales["FN"])
)

print(totales)

print("\nEventos GT evaluados por enfoque:")
print(
    df_detalle_gt
    .groupby("enfoque")
    .size()
)

print("\nEvaluación de los 18 videos: OK")

=== MÉTRICAS POR VIDEO ===
participante             video condicion enfoque  TP  FP  FN     F1
         P01  V1_video1_normal    normal      O1   5   0   0 1.0000
         P01  V1_video1_normal    normal      O2   5   0   0 1.0000
         P01 V1_video2_pocaluz  poca_luz      O1   5   2   0 0.8333
         P01 V1_video2_pocaluz  poca_luz      O2   5   1   0 0.9091
         P02  V2_video1_normal    normal      O1   4   3   0 0.7273
         P02  V2_video1_normal    normal      O2   4   2   0 0.8000
         P02  V2_video2_lentes    lentes      O1   4   6   0 0.5714
         P02  V2_video2_lentes    lentes      O2   4   6   0 0.5714
         P03  V3_video1_normal    normal      O1  12   7   0 0.7742
         P03  V3_video1_normal    normal      O2  12   7   0 0.7742
         P03  V3_video2_lentes    lentes      O1   9   8   0 0.6923
         P03  V3_video2_lentes    lentes      O2   9   6   0 0.7500
         P04  V4_video1_normal    normal      O1   3   4   0 0.6000
         P04  V4_vide

In [19]:
# FASE 6 - PASO 17: TOTALES GLOBALES O1 VS O2

totales = (
    df_metricas_video
    .groupby("enfoque")[["TP","FP","FN"]]
    .sum()
)

totales["F1_micro"] = (
    2*totales["TP"] /
    (2*totales["TP"] + totales["FP"] + totales["FN"])
)

print("=== RESULTADOS GLOBALES PRELIMINARES ===")
print(totales.to_string(
    formatters={
        "F1_micro": lambda x: f"{x:.4f}"
    }
))

print("\nEventos GT evaluados:")
print(df_detalle_gt.groupby("enfoque").size())

=== RESULTADOS GLOBALES PRELIMINARES ===
         TP  FP  FN F1_micro
enfoque                     
O1       80  78   3   0.6639
O2       70  61  13   0.6542

Eventos GT evaluados:
enfoque
O1    83
O2    83
dtype: int64


In [20]:
# FASE 6 - PASO 18: F1 POR PARTICIPANTE

df_participante = (
    df_metricas_video
    .groupby(["participante","enfoque"])[["TP","FP","FN"]]
    .sum()
    .reset_index()
)

df_participante["F1"] = (
    2*df_participante["TP"] /
    (2*df_participante["TP"] + df_participante["FP"] + df_participante["FN"])
)

tabla_ce1 = df_participante.pivot(
    index="participante",
    columns="enfoque",
    values="F1"
).reset_index()

tabla_ce1["diferencia_O2_O1"] = tabla_ce1["O2"] - tabla_ce1["O1"]

print("=== F1 POR PARTICIPANTE ===")
print(
    tabla_ce1.to_string(
        index=False,
        formatters={
            "O1": lambda x: f"{x:.4f}",
            "O2": lambda x: f"{x:.4f}",
            "diferencia_O2_O1": lambda x: f"{x:+.4f}"
        }
    )
)

print("\n=== RESUMEN ===")
print("Media F1 O1:", round(tabla_ce1["O1"].mean(),4))
print("Media F1 O2:", round(tabla_ce1["O2"].mean(),4))
print("Media diferencia O2-O1:", round(tabla_ce1["diferencia_O2_O1"].mean(),4))
print("Participantes donde O2 > O1:", int((tabla_ce1["diferencia_O2_O1"] > 0).sum()))
print("Participantes donde O2 = O1:", int((tabla_ce1["diferencia_O2_O1"] == 0).sum()))
print("Participantes donde O2 < O1:", int((tabla_ce1["diferencia_O2_O1"] < 0).sum()))

tabla_ce1.to_csv(
    CARPETA_FASE6/"f1_por_participante_ce1.csv",
    index=False
)

print("\nTabla CE1 guardada: OK")

=== F1 POR PARTICIPANTE ===
participante     O1     O2 diferencia_O2_O1
         P01 0.9091 0.9524          +0.0433
         P02 0.6400 0.6667          +0.0267
         P03 0.7368 0.7636          +0.0268
         P04 0.5556 0.1538          -0.4017
         P05 0.6154 0.6667          +0.0513
         P06 0.7097 0.6667          -0.0430
         P07 0.6667 0.6667          +0.0000
         P08 0.3529 0.4000          +0.0471
         P09 0.7826 0.6316          -0.1510

=== RESUMEN ===
Media F1 O1: 0.6632
Media F1 O2: 0.6187
Media diferencia O2-O1: -0.0445
Participantes donde O2 > O1: 5
Participantes donde O2 = O1: 1
Participantes donde O2 < O1: 3

Tabla CE1 guardada: OK


In [21]:
# FASE 6 - PASO 19: PRUEBA PAREADA CE1

from scipy.stats import wilcoxon

o1 = tabla_ce1["O1"].to_numpy()
o2 = tabla_ce1["O2"].to_numpy()

resultado_w = wilcoxon(
    o2,
    o1,
    alternative="greater",
    zero_method="wilcox",
    method="auto"
)

print("=== PRUEBA DE WILCOXON CE1 ===")
print("Hipótesis alternativa: F1(O2) > F1(O1)")
print("Participantes:", len(tabla_ce1))
print("Diferencias no nulas:", int((tabla_ce1["diferencia_O2_O1"] != 0).sum()))
print("Estadístico W:", round(float(resultado_w.statistic),4))
print("p unilateral:", round(float(resultado_w.pvalue),6))

if resultado_w.pvalue < 0.05:
    print("\nResultado: diferencia estadísticamente significativa a favor de O2.")
else:
    print("\nResultado: no hay evidencia estadísticamente significativa de que O2 supere a O1.")

print("\nCriterio CE1:", "CUMPLE" if resultado_w.pvalue < 0.05 else "NO CUMPLE")

=== PRUEBA DE WILCOXON CE1 ===
Hipótesis alternativa: F1(O2) > F1(O1)
Participantes: 9
Diferencias no nulas: 8
Estadístico W: 18.0
p unilateral: 0.527344

Resultado: no hay evidencia estadísticamente significativa de que O2 supere a O1.

Criterio CE1: NO CUMPLE


In [22]:
# FASE 6 - PASO 20: TAMAÑO DEL EFECTO CE1

from scipy.stats import rankdata
import numpy as np

diferencias = tabla_ce1["diferencia_O2_O1"].to_numpy()

# Excluir empates exactos para Wilcoxon
dif_no_cero = diferencias[diferencias != 0]

rangos = rankdata(
    np.abs(dif_no_cero),
    method="average"
)

W_pos = rangos[dif_no_cero > 0].sum()
W_neg = rangos[dif_no_cero < 0].sum()

r_rb = (
    (W_pos - W_neg) /
    (W_pos + W_neg)
)

media_diff = diferencias.mean()
mediana_diff = np.median(diferencias)

print("=== TAMAÑO DEL EFECTO CE1 ===")
print("W+:", round(float(W_pos),4))
print("W-:", round(float(W_neg),4))
print("Correlación rank-biserial:", round(float(r_rb),4))
print("Media diferencia O2-O1:", round(float(media_diff),4))
print("Mediana diferencia O2-O1:", round(float(mediana_diff),4))

if r_rb > 0:
    direccion = "a favor de O2"
elif r_rb < 0:
    direccion = "a favor de O1"
else:
    direccion = "sin dirección neta"

print("Dirección del efecto:", direccion)

=== TAMAÑO DEL EFECTO CE1 ===
W+: 18.0
W-: 18.0
Correlación rank-biserial: 0.0
Media diferencia O2-O1: -0.0445
Mediana diferencia O2-O1: 0.0267
Dirección del efecto: sin dirección neta


In [23]:
# FASE 6 - PASO 21: EVALUACIÓN FORMAL DE CE2

total_frames = df_resumen_fase6["frames_procesados"].sum()
total_tiempo = df_resumen_fase6["tiempo_procesamiento_s"].sum()

fps_global = total_frames / total_tiempo
latencia_global_ms = (total_tiempo / total_frames) * 1000

df_ce2 = df_resumen_fase6[
    ["video","condicion","frames_procesados",
     "fps_procesamiento","latencia_media_ms"]
].copy()

df_ce2["cumple_fps"] = df_ce2["fps_procesamiento"] >= 15
df_ce2["cumple_latencia"] = df_ce2["latencia_media_ms"] <= 66.7
df_ce2["cumple_ce2"] = df_ce2["cumple_fps"] & df_ce2["cumple_latencia"]

print("=== CE2 - RENDIMIENTO EN CPU ===")
print("Frames evaluados:", int(total_frames))
print("Tiempo total:", round(total_tiempo,2), "s")
print("FPS global:", round(fps_global,2))
print("Latencia global:", round(latencia_global_ms,2), "ms/frame")

print("\nRango por video:")
print("FPS mínimo:", round(df_ce2["fps_procesamiento"].min(),2))
print("FPS máximo:", round(df_ce2["fps_procesamiento"].max(),2))
print("Latencia mínima:", round(df_ce2["latencia_media_ms"].min(),2), "ms")
print("Latencia máxima:", round(df_ce2["latencia_media_ms"].max(),2), "ms")

print("\nCumplimiento individual:")
print("Videos con FPS >= 15:", int(df_ce2["cumple_fps"].sum()), "/ 18")
print("Videos con latencia <= 66.7 ms:", int(df_ce2["cumple_latencia"].sum()), "/ 18")
print("Videos que cumplen ambos:", int(df_ce2["cumple_ce2"].sum()), "/ 18")

cumple_ce2 = (
    fps_global >= 15 and
    latencia_global_ms <= 66.7
)

print("\nCriterios:")
print("FPS >= 15:", "CUMPLE" if fps_global >= 15 else "NO CUMPLE")
print("Latencia <= 66.7 ms:", "CUMPLE" if latencia_global_ms <= 66.7 else "NO CUMPLE")
print("CE2:", "CUMPLE" if cumple_ce2 else "NO CUMPLE")

df_ce2.to_csv(
    CARPETA_FASE6/"ce2_rendimiento_cpu.csv",
    index=False
)

=== CE2 - RENDIMIENTO EN CPU ===
Frames evaluados: 34862
Tiempo total: 1762.4 s
FPS global: 19.78
Latencia global: 50.55 ms/frame

Rango por video:
FPS mínimo: 17.54
FPS máximo: 23.95
Latencia mínima: 41.76 ms
Latencia máxima: 57.02 ms

Cumplimiento individual:
Videos con FPS >= 15: 18 / 18
Videos con latencia <= 66.7 ms: 18 / 18
Videos que cumplen ambos: 18 / 18

Criterios:
FPS >= 15: CUMPLE
Latencia <= 66.7 ms: CUMPLE
CE2: CUMPLE


In [24]:
# FASE 6 - PASO 22: TABLA DE ROBUSTEZ CE3

filas_ce3 = []

for participante in sorted(df_g["participante"].unique()):
    videos_p = df_g[df_g["participante"] == participante]

    video_normal = videos_p[videos_p["condicion"] == "normal"]["video"].iloc[0]
    fila_adversa = videos_p[videos_p["condicion"] != "normal"].iloc[0]

    video_adverso = fila_adversa["video"]
    condicion_adversa = fila_adversa["condicion"]

    for enfoque in ["O1","O2"]:
        f1_normal = df_metricas_video[
            (df_metricas_video["video"] == video_normal) &
            (df_metricas_video["enfoque"] == enfoque)
        ]["F1"].iloc[0]

        f1_adverso = df_metricas_video[
            (df_metricas_video["video"] == video_adverso) &
            (df_metricas_video["enfoque"] == enfoque)
        ]["F1"].iloc[0]

        filas_ce3.append({
            "participante": participante,
            "condicion_adversa": condicion_adversa,
            "enfoque": enfoque,
            "F1_normal": f1_normal,
            "F1_adverso": f1_adverso,
            "caida_F1": f1_normal - f1_adverso
        })

df_ce3 = pd.DataFrame(filas_ce3)

tabla_ce3 = df_ce3.pivot(
    index=["participante","condicion_adversa"],
    columns="enfoque",
    values=["F1_normal","F1_adverso","caida_F1"]
).reset_index()

tabla_ce3.columns = [
    "_".join([str(x) for x in col if x != ""]).rstrip("_")
    if isinstance(col, tuple) else col
    for col in tabla_ce3.columns
]

tabla_ce3["ventaja_robustez_O2"] = (
    tabla_ce3["caida_F1_O1"] -
    tabla_ce3["caida_F1_O2"]
)

print("=== CE3 - ROBUSTEZ POR PARTICIPANTE ===")
print(
    tabla_ce3.to_string(
        index=False,
        formatters={
            "F1_normal_O1": lambda x: f"{x:.4f}",
            "F1_normal_O2": lambda x: f"{x:.4f}",
            "F1_adverso_O1": lambda x: f"{x:.4f}",
            "F1_adverso_O2": lambda x: f"{x:.4f}",
            "caida_F1_O1": lambda x: f"{x:+.4f}",
            "caida_F1_O2": lambda x: f"{x:+.4f}",
            "ventaja_robustez_O2": lambda x: f"{x:+.4f}"
        }
    )
)

print("\nSubgrupo poca luz:",
      (tabla_ce3["condicion_adversa"] == "poca_luz").sum())

print("Subgrupo lentes:",
      (tabla_ce3["condicion_adversa"] == "lentes").sum())

tabla_ce3.to_csv(
    CARPETA_FASE6/"ce3_robustez_por_participante.csv",
    index=False
)

=== CE3 - ROBUSTEZ POR PARTICIPANTE ===
participante condicion_adversa F1_normal_O1 F1_normal_O2 F1_adverso_O1 F1_adverso_O2 caida_F1_O1 caida_F1_O2 ventaja_robustez_O2
         P01          poca_luz       1.0000       1.0000        0.8333        0.9091     +0.1667     +0.0909             +0.0758
         P02            lentes       0.7273       0.8000        0.5714        0.5714     +0.1558     +0.2286             -0.0727
         P03            lentes       0.7742       0.7742        0.6923        0.7500     +0.0819     +0.0242             +0.0577
         P04          poca_luz       0.6000       0.0000        0.5000        0.3333     +0.1000     -0.3333             +0.4333
         P05          poca_luz       0.5714       0.8571        0.6667        0.0000     -0.0952     +0.8571             -0.9524
         P06          poca_luz       0.6667       0.6667        0.7500        0.6667     -0.0833     +0.0000             -0.0833
         P07          poca_luz       0.8571       1.0000 

In [25]:
# FASE 6 - PASO 23: CE3 INFERENCIAL - POCA LUZ

from scipy.stats import shapiro, ttest_1samp, wilcoxon
import numpy as np

ce3_luz = tabla_ce3[
    tabla_ce3["condicion_adversa"] == "poca_luz"
].copy()

ventaja = ce3_luz["ventaja_robustez_O2"].to_numpy()

# Normalidad de las diferencias
shapiro_ce3 = shapiro(ventaja)

print("=== CE3 - POCA LUZ (n=6) ===")
print("Hipótesis: la caída de F1 de O2 es menor que la de O1")
print("Equivalente: caída_O1 - caída_O2 > 0")

print("\nShapiro-Wilk:")
print("W:", round(float(shapiro_ce3.statistic),4))
print("p:", round(float(shapiro_ce3.pvalue),6))

# Elegir prueba según normalidad
if shapiro_ce3.pvalue >= 0.05:
    prueba = "t pareada unilateral"
    resultado = ttest_1samp(
        ventaja,
        popmean=0,
        alternative="greater"
    )
    estadistico = resultado.statistic
    pvalor = resultado.pvalue
else:
    prueba = "Wilcoxon unilateral"
    resultado = wilcoxon(
        ventaja,
        alternative="greater",
        zero_method="wilcox",
        method="auto"
    )
    estadistico = resultado.statistic
    pvalor = resultado.pvalue

print("\nPrueba seleccionada:", prueba)
print("Estadístico:", round(float(estadistico),4))
print("p unilateral:", round(float(pvalor),6))

print("\nMedia caída O1:", round(ce3_luz["caida_F1_O1"].mean(),4))
print("Media caída O2:", round(ce3_luz["caida_F1_O2"].mean(),4))
print("Media ventaja O2:", round(ventaja.mean(),4))

cumple_ce3_luz = pvalor < 0.05

print(
    "\nResultado:",
    "O2 presenta una caída significativamente menor que O1."
    if cumple_ce3_luz
    else "No hay evidencia de que O2 presente una caída significativamente menor que O1."
)

print("CE3 - poca luz:", "CUMPLE" if cumple_ce3_luz else "NO CUMPLE")

=== CE3 - POCA LUZ (n=6) ===
Hipótesis: la caída de F1 de O2 es menor que la de O1
Equivalente: caída_O1 - caída_O2 > 0

Shapiro-Wilk:
W: 0.8878
p: 0.307024

Prueba seleccionada: t pareada unilateral
Estadístico: -0.2168
p unilateral: 0.58152

Media caída O1: 0.049
Media caída O2: 0.0945
Media ventaja O2: -0.0456

Resultado: No hay evidencia de que O2 presente una caída significativamente menor que O1.
CE3 - poca luz: NO CUMPLE


In [26]:
# FASE 6 - PASO 24: CE3 DESCRIPTIVO - LENTES

ce3_lentes = tabla_ce3[
    tabla_ce3["condicion_adversa"] == "lentes"
].copy()

print("=== CE3 - LENTES (n=3) ===")
print(
    ce3_lentes[
        ["participante","F1_normal_O1","F1_adverso_O1",
         "caida_F1_O1","F1_normal_O2","F1_adverso_O2",
         "caida_F1_O2","ventaja_robustez_O2"]
    ].to_string(
        index=False,
        formatters={
            "F1_normal_O1": lambda x: f"{x:.4f}",
            "F1_adverso_O1": lambda x: f"{x:.4f}",
            "caida_F1_O1": lambda x: f"{x:+.4f}",
            "F1_normal_O2": lambda x: f"{x:.4f}",
            "F1_adverso_O2": lambda x: f"{x:.4f}",
            "caida_F1_O2": lambda x: f"{x:+.4f}",
            "ventaja_robustez_O2": lambda x: f"{x:+.4f}"
        }
    )
)

for enfoque in ["O1","O2"]:
    valores = ce3_lentes[f"caida_F1_{enfoque}"]
    print(f"\n{enfoque}:")
    print("Media caída:", round(valores.mean(),4))
    print("Mediana:", round(valores.median(),4))
    print("Mínimo:", round(valores.min(),4))
    print("Máximo:", round(valores.max(),4))
    print("Rango:", round(valores.max()-valores.min(),4))

print("\nVentaja media de robustez O2:",
      round(ce3_lentes["ventaja_robustez_O2"].mean(),4))

print("Casos O2 más robusto:",
      int((ce3_lentes["ventaja_robustez_O2"] > 0).sum()), "/ 3")

print("Casos O1 más robusto:",
      int((ce3_lentes["ventaja_robustez_O2"] < 0).sum()), "/ 3")

=== CE3 - LENTES (n=3) ===
participante F1_normal_O1 F1_adverso_O1 caida_F1_O1 F1_normal_O2 F1_adverso_O2 caida_F1_O2 ventaja_robustez_O2
         P02       0.7273        0.5714     +0.1558       0.8000        0.5714     +0.2286             -0.0727
         P03       0.7742        0.6923     +0.0819       0.7742        0.7500     +0.0242             +0.0577
         P08       0.2353        0.4706     -0.2353       0.3333        0.4615     -0.1282             -0.1071

O1:
Media caída: 0.0008
Mediana: 0.0819
Mínimo: -0.2353
Máximo: 0.1558
Rango: 0.3911

O2:
Media caída: 0.0415
Mediana: 0.0242
Mínimo: -0.1282
Máximo: 0.2286
Rango: 0.3568

Ventaja media de robustez O2: -0.0407
Casos O2 más robusto: 1 / 3
Casos O1 más robusto: 2 / 3


In [27]:
# FASE 6 - PASO 25: CE4 - TIEMPO HASTA LA ALERTA

filas_ce4 = []

for _, fila in df_g.iterrows():
    video = fila["video"]
    participante = fila["participante"]
    condicion = fila["condicion"]
    fps = float(fila["fps"])

    ruta_alertas = CARPETA_FASE6/f"alertas_{video}.csv"

    try:
        alertas = pd.read_csv(ruta_alertas)
    except pd.errors.EmptyDataError:
        alertas = pd.DataFrame(columns=["enfoque","evento","frame_alerta"])

    gt_video = df_gt[df_gt["video"] == video].copy()

    for enfoque in ["O1","O2"]:
        for evento in ["cierre_ocular","bostezo"]:
            gt_tipo = gt_video[
                gt_video["etiqueta"] == evento
            ].sort_values("frame_inicio_final")

            if len(alertas):
                al_tipo = alertas[
                    (alertas["enfoque"] == enfoque) &
                    (alertas["evento"] == evento)
                ].sort_values("frame_alerta")
            else:
                al_tipo = pd.DataFrame(columns=["frame_alerta"])

            for _, gt_evento in gt_tipo.iterrows():
                inicio = int(gt_evento["frame_inicio_final"])
                fin = int(gt_evento["frame_fin_final"])

                candidatos = al_tipo[
                    (al_tipo["frame_alerta"] >= inicio) &
                    (al_tipo["frame_alerta"] <= fin)
                ]

                if len(candidatos):
                    frame_alerta = int(candidatos["frame_alerta"].min())
                    retraso_frames = frame_alerta - inicio
                    retraso_s = retraso_frames/fps
                    detectado = 1
                else:
                    frame_alerta = pd.NA
                    retraso_frames = pd.NA
                    retraso_s = np.nan
                    detectado = 0

                filas_ce4.append({
                    "participante": participante,
                    "video": video,
                    "condicion": condicion,
                    "enfoque": enfoque,
                    "evento": evento,
                    "frame_inicio_gt": inicio,
                    "frame_fin_gt": fin,
                    "frame_alerta": frame_alerta,
                    "detectado": detectado,
                    "retraso_frames": retraso_frames,
                    "retraso_s": retraso_s
                })

df_ce4 = pd.DataFrame(filas_ce4)

df_ce4.to_csv(
    CARPETA_FASE6/"ce4_tiempos_alerta.csv",
    index=False
)

print("=== CE4 - TIEMPO HASTA LA ALERTA ===")

resumen_ce4 = (
    df_ce4[df_ce4["detectado"] == 1]
    .groupby(["enfoque","evento"])["retraso_s"]
    .agg(["count","mean","median","min","max"])
)

print(
    resumen_ce4.to_string(
        formatters={
            "mean": lambda x: f"{x:.3f}",
            "median": lambda x: f"{x:.3f}",
            "min": lambda x: f"{x:.3f}",
            "max": lambda x: f"{x:.3f}"
        }
    )
)

print("\nEventos detectados:")
print(
    df_ce4.groupby("enfoque")["detectado"]
    .agg(["sum","count"])
)

print("\nCE4 calculado y guardado: OK")

=== CE4 - TIEMPO HASTA LA ALERTA ===
                       count  mean median   min   max
enfoque evento                                       
O1      bostezo           26 2.587  2.780 0.733 3.133
        cierre_ocular     54 0.721  0.567 0.033 4.499
O2      bostezo           22 2.907  2.967 2.167 3.800
        cierre_ocular     48 1.210  0.883 0.133 4.200

Eventos detectados:
         sum  count
enfoque            
O1        80     83
O2        70     83

CE4 calculado y guardado: OK


In [28]:
# FASE 6 - PASO 26: SHAPIRO + BOOTSTRAP CE1

from scipy.stats import shapiro
import numpy as np

diferencias = tabla_ce1["diferencia_O2_O1"].to_numpy()

# Normalidad
sh = shapiro(diferencias)

# Bootstrap pareado de la diferencia media
rng = np.random.default_rng(42)
n_boot = 10000
boot_medias = []

for _ in range(n_boot):
    muestra = rng.choice(diferencias, size=len(diferencias), replace=True)
    boot_medias.append(muestra.mean())

ic_inf, ic_sup = np.percentile(boot_medias, [2.5,97.5])

print("=== COMPLEMENTO ESTADÍSTICO CE1 ===")
print("Shapiro-Wilk W:", round(float(sh.statistic),4))
print("Shapiro-Wilk p:", round(float(sh.pvalue),6))

if sh.pvalue < 0.05:
    print("Normalidad: NO")
    print("Prueba correspondiente: Wilcoxon pareado")
else:
    print("Normalidad: SÍ")
    print("Prueba correspondiente: t pareada")

print("\nDiferencia media O2-O1:", round(float(diferencias.mean()),4))
print("IC bootstrap 95%:", f"[{ic_inf:.4f}, {ic_sup:.4f}]")
print("Remuestreos:", n_boot)
print("Semilla:", 42)

=== COMPLEMENTO ESTADÍSTICO CE1 ===
Shapiro-Wilk W: 0.6968
Shapiro-Wilk p: 0.001304
Normalidad: NO
Prueba correspondiente: Wilcoxon pareado

Diferencia media O2-O1: -0.0445
IC bootstrap 95%: [-0.1492, 0.0299]
Remuestreos: 10000
Semilla: 42


In [29]:
# FASE 6 - PASO 27: RESUMEN FINAL DE CRITERIOS

resumen_criterios = pd.DataFrame([
    {
        "criterio": "CE1",
        "descripcion": "Mejora del desempeño O2 frente a O1",
        "resultado_principal": "Wilcoxon W=18.0; p=0.527344",
        "resultado": "NO CUMPLE"
    },
    {
        "criterio": "CE2",
        "descripcion": "Eficiencia en tiempo real solo CPU",
        "resultado_principal": f"{fps_global:.2f} FPS; {latencia_global_ms:.2f} ms/frame",
        "resultado": "CUMPLE"
    },
    {
        "criterio": "CE3",
        "descripcion": "Robustez ante poca luz",
        "resultado_principal": "t unilateral=-0.2168; p=0.581520",
        "resultado": "NO CUMPLE"
    },
    {
        "criterio": "CE3-L",
        "descripcion": "Robustez ante lentes (descriptivo)",
        "resultado_principal": "O2 más robusto en 1/3; O1 en 2/3",
        "resultado": "DESCRIPTIVO"
    },
    {
        "criterio": "CE4",
        "descripcion": "Tiempo desde inicio del evento hasta alerta",
        "resultado_principal": "Evaluación descriptiva",
        "resultado": "REPORTADO"
    }
])

resumen_criterios.to_csv(
    CARPETA_FASE6/"resumen_criterios_fase6.csv",
    index=False,
    encoding="utf-8-sig"
)

print("=== RESUMEN FINAL FASE 6 ===")
print(resumen_criterios.to_string(index=False))

print("\n=== DATOS CLAVE CE1 ===")
print("F1 medio O1:", round(tabla_ce1["O1"].mean(),4))
print("F1 medio O2:", round(tabla_ce1["O2"].mean(),4))
print("Diferencia media:", round(tabla_ce1["diferencia_O2_O1"].mean(),4))
print("Wilcoxon p:", 0.527344)
print("Rank-biserial:", 0.0)
print("IC bootstrap 95%:", f"[{ic_inf:.4f}, {ic_sup:.4f}]")

print("\n=== DATOS CLAVE CE2 ===")
print("FPS global:", round(fps_global,2))
print("Latencia global:", round(latencia_global_ms,2), "ms/frame")
print("Videos que cumplen:", int(df_ce2["cumple_ce2"].sum()), "/ 18")

print("\n=== DATOS CLAVE CE3 ===")
print("Poca luz: NO CUMPLE")
print("p =", 0.581520)
print("Lentes: análisis descriptivo")

print("\n=== DATOS CLAVE CE4 ===")
print(resumen_ce4.to_string())

print("\nResumen guardado: OK")

=== RESUMEN FINAL FASE 6 ===
criterio                                 descripcion              resultado_principal   resultado
     CE1         Mejora del desempeño O2 frente a O1      Wilcoxon W=18.0; p=0.527344   NO CUMPLE
     CE2          Eficiencia en tiempo real solo CPU        19.78 FPS; 50.55 ms/frame      CUMPLE
     CE3                      Robustez ante poca luz t unilateral=-0.2168; p=0.581520   NO CUMPLE
   CE3-L          Robustez ante lentes (descriptivo) O2 más robusto en 1/3; O1 en 2/3 DESCRIPTIVO
     CE4 Tiempo desde inicio del evento hasta alerta           Evaluación descriptiva   REPORTADO

=== DATOS CLAVE CE1 ===
F1 medio O1: 0.6632
F1 medio O2: 0.6187
Diferencia media: -0.0445
Wilcoxon p: 0.527344
Rank-biserial: 0.0
IC bootstrap 95%: [-0.1492, 0.0299]

=== DATOS CLAVE CE2 ===
FPS global: 19.78
Latencia global: 50.55 ms/frame
Videos que cumplen: 18 / 18

=== DATOS CLAVE CE3 ===
Poca luz: NO CUMPLE
p = 0.58152
Lentes: análisis descriptivo

=== DATOS CLAVE CE4 ===
  

In [30]:
# FASE 6 - PASO 28: RESULTADOS POR EVENTO Y POR CONDICIÓN

def resumir_metricas(grupos):
    claves = ["enfoque"] + grupos

    tp_fn = (
        df_detalle_gt
        .groupby(claves + ["resultado"])
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )

    if "TP" not in tp_fn.columns:
        tp_fn["TP"] = 0
    if "FN" not in tp_fn.columns:
        tp_fn["FN"] = 0

    fp = (
        df_detalle_alertas[
            df_detalle_alertas["clasificacion"] == "FP"
        ]
        .groupby(claves)
        .size()
        .reset_index(name="FP")
    )

    resumen = tp_fn.merge(fp, on=claves, how="left")
    resumen["FP"] = resumen["FP"].fillna(0).astype(int)
    resumen["TP"] = resumen["TP"].astype(int)
    resumen["FN"] = resumen["FN"].astype(int)

    resumen["precision"] = (
        resumen["TP"] /
        (resumen["TP"] + resumen["FP"])
    )

    resumen["recall"] = (
        resumen["TP"] /
        (resumen["TP"] + resumen["FN"])
    )

    resumen["F1"] = (
        2*resumen["TP"] /
        (2*resumen["TP"] + resumen["FP"] + resumen["FN"])
    )

    return resumen


# ------------------------------------------------------------
# RESULTADOS POR TIPO DE EVENTO
# ------------------------------------------------------------

df_resultados_evento = resumir_metricas(["evento"])

print("=== RESULTADOS POR TIPO DE EVENTO ===")
print(
    df_resultados_evento[
        ["enfoque","evento","TP","FP","FN","precision","recall","F1"]
    ].to_string(
        index=False,
        formatters={
            "precision": lambda x: f"{x:.4f}",
            "recall": lambda x: f"{x:.4f}",
            "F1": lambda x: f"{x:.4f}"
        }
    )
)


# ------------------------------------------------------------
# RESULTADOS POR CONDICIÓN
# ------------------------------------------------------------

df_resultados_condicion = resumir_metricas(["condicion"])

orden_condiciones = {
    "normal": 0,
    "poca_luz": 1,
    "lentes": 2
}

df_resultados_condicion["orden"] = (
    df_resultados_condicion["condicion"]
    .map(orden_condiciones)
)

df_resultados_condicion = (
    df_resultados_condicion
    .sort_values(["enfoque","orden"])
    .drop(columns="orden")
    .reset_index(drop=True)
)

print("\n=== RESULTADOS POR CONDICIÓN ===")
print(
    df_resultados_condicion[
        ["enfoque","condicion","TP","FP","FN","precision","recall","F1"]
    ].to_string(
        index=False,
        formatters={
            "precision": lambda x: f"{x:.4f}",
            "recall": lambda x: f"{x:.4f}",
            "F1": lambda x: f"{x:.4f}"
        }
    )
)


# ------------------------------------------------------------
# GUARDAR
# ------------------------------------------------------------

df_resultados_evento.to_csv(
    CARPETA_FASE6/"resultados_por_evento.csv",
    index=False,
    encoding="utf-8-sig"
)

df_resultados_condicion.to_csv(
    CARPETA_FASE6/"resultados_por_condicion.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nTablas finales guardadas: OK")

=== RESULTADOS POR TIPO DE EVENTO ===
enfoque        evento  TP  FP  FN precision recall     F1
     O1       bostezo  26   5   1    0.8387 0.9630 0.8966
     O1 cierre_ocular  54  73   2    0.4252 0.9643 0.5902
     O2       bostezo  22   1   5    0.9565 0.8148 0.8800
     O2 cierre_ocular  48  60   8    0.4444 0.8571 0.5854

=== RESULTADOS POR CONDICIÓN ===
enfoque condicion  TP  FP  FN precision recall     F1
     O1    normal  40  36   3    0.5263 0.9302 0.6723
     O1  poca_luz  23  19   0    0.5476 1.0000 0.7077
     O1    lentes  17  23   0    0.4250 1.0000 0.5965
     O2    normal  34  25   9    0.5763 0.7907 0.6667
     O2  poca_luz  20  18   3    0.5263 0.8696 0.6557
     O2    lentes  16  18   1    0.4706 0.9412 0.6275

Tablas finales guardadas: OK


In [31]:
# FASE 6 - PASO 29: COMPARACIÓN DESCRIPTIVA O1 VS O2

evento_o1 = df_resultados_evento[
    df_resultados_evento["enfoque"] == "O1"
].set_index("evento")

evento_o2 = df_resultados_evento[
    df_resultados_evento["enfoque"] == "O2"
].set_index("evento")

comparacion_evento = pd.DataFrame({
    "F1_O1": evento_o1["F1"],
    "F1_O2": evento_o2["F1"],
    "diferencia_O2_O1": evento_o2["F1"] - evento_o1["F1"],
    "precision_O1": evento_o1["precision"],
    "precision_O2": evento_o2["precision"],
    "recall_O1": evento_o1["recall"],
    "recall_O2": evento_o2["recall"]
}).reset_index()

cond_o1 = df_resultados_condicion[
    df_resultados_condicion["enfoque"] == "O1"
].set_index("condicion")

cond_o2 = df_resultados_condicion[
    df_resultados_condicion["enfoque"] == "O2"
].set_index("condicion")

comparacion_condicion = pd.DataFrame({
    "F1_O1": cond_o1["F1"],
    "F1_O2": cond_o2["F1"],
    "diferencia_O2_O1": cond_o2["F1"] - cond_o1["F1"],
    "precision_O1": cond_o1["precision"],
    "precision_O2": cond_o2["precision"],
    "recall_O1": cond_o1["recall"],
    "recall_O2": cond_o2["recall"]
}).reset_index()

print("=== COMPARACIÓN POR EVENTO ===")
print(comparacion_evento.to_string(
    index=False,
    formatters={
        "F1_O1": lambda x:f"{x:.4f}",
        "F1_O2": lambda x:f"{x:.4f}",
        "diferencia_O2_O1": lambda x:f"{x:+.4f}",
        "precision_O1": lambda x:f"{x:.4f}",
        "precision_O2": lambda x:f"{x:.4f}",
        "recall_O1": lambda x:f"{x:.4f}",
        "recall_O2": lambda x:f"{x:.4f}"
    }
))

print("\n=== COMPARACIÓN POR CONDICIÓN ===")
print(comparacion_condicion.to_string(
    index=False,
    formatters={
        "F1_O1": lambda x:f"{x:.4f}",
        "F1_O2": lambda x:f"{x:.4f}",
        "diferencia_O2_O1": lambda x:f"{x:+.4f}",
        "precision_O1": lambda x:f"{x:.4f}",
        "precision_O2": lambda x:f"{x:.4f}",
        "recall_O1": lambda x:f"{x:.4f}",
        "recall_O2": lambda x:f"{x:.4f}"
    }
))

comparacion_evento.to_csv(
    CARPETA_FASE6/"comparacion_O1_O2_por_evento.csv",
    index=False,
    encoding="utf-8-sig"
)

comparacion_condicion.to_csv(
    CARPETA_FASE6/"comparacion_O1_O2_por_condicion.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nComparaciones guardadas: OK")

=== COMPARACIÓN POR EVENTO ===
       evento  F1_O1  F1_O2 diferencia_O2_O1 precision_O1 precision_O2 recall_O1 recall_O2
      bostezo 0.8966 0.8800          -0.0166       0.8387       0.9565    0.9630    0.8148
cierre_ocular 0.5902 0.5854          -0.0048       0.4252       0.4444    0.9643    0.8571

=== COMPARACIÓN POR CONDICIÓN ===
condicion  F1_O1  F1_O2 diferencia_O2_O1 precision_O1 precision_O2 recall_O1 recall_O2
   normal 0.6723 0.6667          -0.0056       0.5263       0.5763    0.9302    0.7907
 poca_luz 0.7077 0.6557          -0.0520       0.5476       0.5263    1.0000    0.8696
   lentes 0.5965 0.6275          +0.0310       0.4250       0.4706    1.0000    0.9412

Comparaciones guardadas: OK


In [32]:
# FASE 6 - PASO 30: FALSAS ALERTAS/MIN Y FALLOS DE ROSTRO

duracion_por_video = df_g.set_index("video")["duracion_s"].to_dict()

filas_fp = []
for enfoque in ["O1","O2"]:
    for condicion in ["normal","poca_luz","lentes"]:
        videos_cond = df_g[df_g["condicion"] == condicion]["video"].tolist()
        duracion_min = sum(duracion_por_video[v] for v in videos_cond)/60

        fp = df_detalle_alertas[
            (df_detalle_alertas["enfoque"] == enfoque) &
            (df_detalle_alertas["condicion"] == condicion) &
            (df_detalle_alertas["clasificacion"] == "FP")
        ].shape[0]

        filas_fp.append({
            "enfoque": enfoque,
            "condicion": condicion,
            "FP": fp,
            "duracion_min": duracion_min,
            "falsas_alertas_min": fp/duracion_min
        })

df_fp_min = pd.DataFrame(filas_fp)

# Tasa de fallo facial ponderada por frames
df_fallo_cond = (
    df_resumen_fase6
    .groupby("condicion")[["frames_procesados","fallos_rostro"]]
    .sum()
    .reset_index()
)

df_fallo_cond["tasa_fallo_rostro_pct"] = (
    100*df_fallo_cond["fallos_rostro"] /
    df_fallo_cond["frames_procesados"]
)

print("=== FALSAS ALERTAS POR MINUTO ===")
print(df_fp_min.to_string(
    index=False,
    formatters={
        "duracion_min": lambda x:f"{x:.2f}",
        "falsas_alertas_min": lambda x:f"{x:.3f}"
    }
))

print("\n=== FALLO DE DETECCIÓN FACIAL POR CONDICIÓN ===")
print(df_fallo_cond.to_string(
    index=False,
    formatters={"tasa_fallo_rostro_pct": lambda x:f"{x:.4f}%"}
))

df_fp_min.to_csv(
    CARPETA_FASE6/"falsas_alertas_por_minuto.csv",
    index=False,
    encoding="utf-8-sig"
)

df_fallo_cond.to_csv(
    CARPETA_FASE6/"fallo_rostro_por_condicion.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nIndicadores adicionales guardados: OK")

=== FALSAS ALERTAS POR MINUTO ===
enfoque condicion  FP duracion_min falsas_alertas_min
     O1    normal  36        10.19              3.533
     O1  poca_luz  19         5.54              3.431
     O1    lentes  23         4.19              5.489
     O2    normal  25        10.19              2.454
     O2  poca_luz  18         5.54              3.250
     O2    lentes  18         4.19              4.296

=== FALLO DE DETECCIÓN FACIAL POR CONDICIÓN ===
condicion  frames_procesados  fallos_rostro tasa_fallo_rostro_pct
   lentes               7542              0               0.0000%
   normal              18312              0               0.0000%
 poca_luz               9008              0               0.0000%

Indicadores adicionales guardados: OK


In [33]:
# FASE 6 - PASO 31: PREPARAR BENCHMARK CE2 / RNF08

import os
import platform
import psutil
import onnxruntime as ort

REPETICIONES_CE2 = 3

print("=== CONFIGURACIÓN BENCHMARK CE2 ===")
print("Repeticiones por video:", REPETICIONES_CE2)

print("\n=== EQUIPO ===")
print("Sistema:", platform.system(), platform.release())
print("Procesador:", platform.processor())
print("Núcleos físicos:", psutil.cpu_count(logical=False))
print("Núcleos lógicos:", psutil.cpu_count(logical=True))
print("RAM total:", round(psutil.virtual_memory().total/(1024**3),2), "GB")

print("\n=== ONNX RUNTIME ===")
print("T1:", sesion_t1.get_providers())
print("T2:", sesion_t2.get_providers())

assert sesion_t1.get_providers() == ["CPUExecutionProvider"]
assert sesion_t2.get_providers() == ["CPUExecutionProvider"]

CARPETA_BENCHMARK = CARPETA_FASE6/"benchmark_ce2"
CARPETA_BENCHMARK.mkdir(parents=True, exist_ok=True)

print("\nEjecución CNN exclusivamente en CPU: OK")
print("Carpeta benchmark:", CARPETA_BENCHMARK)

=== CONFIGURACIÓN BENCHMARK CE2 ===
Repeticiones por video: 3

=== EQUIPO ===
Sistema: Windows 10
Procesador: Intel64 Family 6 Model 183 Stepping 1, GenuineIntel
Núcleos físicos: 16
Núcleos lógicos: 24
RAM total: 15.71 GB

=== ONNX RUNTIME ===
T1: ['CPUExecutionProvider']
T2: ['CPUExecutionProvider']

Ejecución CNN exclusivamente en CPU: OK
Carpeta benchmark: ..\results\fase6\benchmark_ce2


In [34]:
# FASE 6 - PASO 32: PRUEBA DE MONITOREO CPU/RAM

import threading
import time
import psutil
import os

def ejecutar_con_monitor(fila_video):
    proceso = psutil.Process(os.getpid())
    detener = threading.Event()
    muestras_cpu = []
    muestras_ram = []

    proceso.cpu_percent(interval=None)

    def monitor():
        while not detener.is_set():
            cpu_raw = proceso.cpu_percent(interval=None)
            ram_mb = proceso.memory_info().rss/(1024**2)

            muestras_cpu.append(cpu_raw)
            muestras_ram.append(ram_mb)

            time.sleep(0.25)

    hilo = threading.Thread(target=monitor)
    hilo.start()

    inicio = time.perf_counter()

    try:
        df_f, df_a, resumen = procesar_video_completo(
            fila_video,
            guardar=False
        )
    finally:
        tiempo_total = time.perf_counter()-inicio
        detener.set()
        hilo.join()

    frames = len(df_f)
    fps_proc = frames/tiempo_total
    latencia_ms = (tiempo_total/frames)*1000

    return {
        "video": fila_video["video"],
        "frames": frames,
        "tiempo_s": tiempo_total,
        "fps": fps_proc,
        "latencia_ms": latencia_ms,
        "cpu_proceso_media_pct": np.mean(muestras_cpu),
        "cpu_proceso_max_pct": np.max(muestras_cpu),
        "ram_media_mb": np.mean(muestras_ram),
        "ram_max_mb": np.max(muestras_ram),
        "muestras": len(muestras_cpu)
    }


# Prueba con un solo video
fila_test = df_g[
    df_g["video"] == "V1_video1_normal"
].iloc[0]

print("=== PRUEBA BENCHMARK ===")
print("Video:", fila_test["video"])

resultado_benchmark_test = ejecutar_con_monitor(
    fila_test
)

print("\n=== RESULTADO ===")
for clave, valor in resultado_benchmark_test.items():
    if isinstance(valor, float):
        print(f"{clave}: {valor:.2f}")
    else:
        print(f"{clave}: {valor}")

assert resultado_benchmark_test["frames"] == int(fila_test["frames"])

print("\nMonitor CPU/RAM funcionando: OK")

=== PRUEBA BENCHMARK ===
Video: V1_video1_normal
V1_video1_normal: 500 frames procesados
V1_video1_normal: 1000 frames procesados
V1_video1_normal: 1500 frames procesados
V1_video1_normal: 2000 frames procesados

=== RESULTADO ===
video: V1_video1_normal
frames: 2340
tiempo_s: 126.93
fps: 18.44
latencia_ms: 54.24
cpu_proceso_media_pct: 2323.58
cpu_proceso_max_pct: 2543.80
ram_media_mb: 375.27
ram_max_mb: 378.66
muestras: 495

Monitor CPU/RAM funcionando: OK


In [1]:
# RECUPERAR ENTORNO SIN REPETIR PROCESAMIENTOS

import json
import re
from pathlib import Path

RUTA_NOTEBOOK = Path("09_fase6_pruebas_G.ipynb")

with open(RUTA_NOTEBOOK, "r", encoding="utf-8") as f:
    nb = json.load(f)

# Solo necesitamos las definiciones.
# Se omiten PASO 8 y PASO 10 porque eran pruebas,
# y PASO 12 en adelante porque procesan/evalúan resultados.
pasos_necesarios = {1,2,3,4,5,6,7,9,11}

ejecutados = []

for celda in nb["cells"]:
    if celda.get("cell_type") != "code":
        continue

    codigo = "".join(celda.get("source", []))
    encontrado = re.search(r"PASO\s+(\d+)", codigo, re.IGNORECASE)

    if encontrado:
        paso = int(encontrado.group(1))

        if paso in pasos_necesarios:
            print(f"Recuperando PASO {paso}...")
            exec(compile(codigo, f"<PASO {paso}>", "exec"), globals())
            ejecutados.append(paso)

print("\nPasos recuperados:", sorted(set(ejecutados)))
print("Entorno base recuperado: OK")

Recuperando PASO 1...
=== FASE 6 - INICIALIZACIÓN ===

Parámetros congelados:
EAR: 0.1404
MAR: 0.1624
Tau ojo O2: 0.580015
Tau bostezo O2: 0.585053
Cierre mínimo: 0.5 s
Bostezo mínimo: 3.0 s
Ventana PERCLOS: 15.0 s

Conjunto G:
Participantes: 9
Videos: 18
condicion
normal      9
poca_luz    6
lentes      3
Name: count, dtype: int64

Ground Truth:
Eventos: 83
etiqueta
cierre_ocular    56
bostezo          27
Name: count, dtype: int64

Carpeta de resultados Fase 6:
..\results\fase6

Inicialización Fase 6: OK
Recuperando PASO 4...
Funciones geométricas y recortes cargados: OK
Margen ojos: 35%
Margen boca: 20%
Recuperando PASO 7...
Función de decisión O1/O2 cargada: OK
O1: EAR/MAR + persistencia
O2: 0.5 CNN + 0.5 geometría + persistencia
PERCLOS: indicador complementario, ventana de 15 s
Recuperando PASO 9...
Función de procesamiento completo por fotograma: OK
MediaPipe → EAR/MAR → recortes → T1/T2 → O1/O2
Recuperando PASO 11...
Función de procesamiento de video completo: OK
Salida: frames 

In [2]:
# VERIFICACIÓN DEL ENTORNO RECUPERADO

print("=== VERIFICACIÓN ENTORNO ===")

variables = [
    "sesion_t1",
    "sesion_t2",
    "face_landmarker",
    "df_g",
    "df_gt",
    "procesar_video_completo",
    "decidir_estado"
]

for v in variables:
    print(
        v,
        "->",
        "OK" if v in globals() else "FALTA"
    )

=== VERIFICACIÓN ENTORNO ===
sesion_t1 -> FALTA
sesion_t2 -> FALTA
face_landmarker -> FALTA
df_g -> OK
df_gt -> OK
procesar_video_completo -> OK
decidir_estado -> FALTA


In [3]:
# RECUPERAR MODELOS + MEDIAPIPE + DECISIÓN O1/O2

import onnxruntime as ort
import mediapipe as mp
from pathlib import Path

print("=== RECUPERACIÓN COMPONENTES FASE 6 ===")

# ------------------------------------------------------------
# RUTAS MODELOS
# ------------------------------------------------------------

MODELO_T1 = Path("../models/mobilenetv2_ojos_v2.onnx")
MODELO_T2 = Path("../models/mobilenetv2_yawn_dedup.onnx")

print("T1 existe:", MODELO_T1.exists())
print("T2 existe:", MODELO_T2.exists())


# ------------------------------------------------------------
# CARGAR ONNX CPU
# ------------------------------------------------------------

sesion_t1 = ort.InferenceSession(
    str(MODELO_T1),
    providers=["CPUExecutionProvider"]
)

sesion_t2 = ort.InferenceSession(
    str(MODELO_T2),
    providers=["CPUExecutionProvider"]
)

print("T1 cargado:", sesion_t1.get_providers())
print("T2 cargado:", sesion_t2.get_providers())


# ------------------------------------------------------------
# CARGAR MEDIAPIPE
# ------------------------------------------------------------

MODELO_FACE = Path("../models/face_landmarker.task")

base_options = mp.tasks.BaseOptions(
    model_asset_path=str(MODELO_FACE)
)

options = mp.tasks.vision.FaceLandmarkerOptions(
    base_options=base_options,
    running_mode=mp.tasks.vision.RunningMode.IMAGE,
    num_faces=1
)

face_landmarker = mp.tasks.vision.FaceLandmarker.create_from_options(
    options
)

print("Face Landmarker cargado: OK")


# ------------------------------------------------------------
# VERIFICACIÓN DECISIÓN
# ------------------------------------------------------------

if "decidir_estado" in globals():
    print("Decisión O1/O2: OK")
else:
    print("Decisión O1/O2: FALTA")

=== RECUPERACIÓN COMPONENTES FASE 6 ===
T1 existe: True
T2 existe: True
T1 cargado: ['CPUExecutionProvider']
T2 cargado: ['CPUExecutionProvider']
Face Landmarker cargado: OK
Decisión O1/O2: FALTA


In [8]:
print("=== VERIFICACIÓN FINAL ENTORNO ===")

for v in [
    "sesion_t1",
    "sesion_t2",
    "face_landmarker",
    "evaluar_frame_t3",
    "procesar_frame_completo",
    "procesar_video_completo"
]:
    print(
        v,
        "->",
        "OK" if v in globals() else "FALTA"
    )

=== VERIFICACIÓN FINAL ENTORNO ===
sesion_t1 -> OK
sesion_t2 -> OK
face_landmarker -> OK
evaluar_frame_t3 -> OK
procesar_frame_completo -> OK
procesar_video_completo -> OK


In [ ]:
# RECUPERAR COMPONENTES BENCHMARK CE2 / RNF08

import threading
import time
import psutil
import os

# Configuración benchmark
REPETICIONES_CE2 = 3

CARPETA_BENCHMARK = CARPETA_RESULTADOS / "fase6" / "benchmark_ce2"
CARPETA_BENCHMARK.mkdir(
    parents=True,
    exist_ok=True
)

def ejecutar_con_monitor(fila_video):
    proceso = psutil.Process(os.getpid())
    detener = threading.Event()
    muestras_cpu = []
    muestras_ram = []
    proceso.cpu_percent(interval=None)
    def monitor():
        while not detener.is_set():

            cpu_raw = proceso.cpu_percent(interval=None)

            ram_mb = (
                proceso.memory_info().rss /
                (1024**2)
            )
            muestras_cpu.append(cpu_raw)
            muestras_ram.append(ram_mb)

            time.sleep(0.25)
    hilo = threading.Thread(
        target=monitor
    )

    hilo.start()
    inicio = time.perf_counter()

    try:

        df_f, df_a, resumen = procesar_video_completo(
            fila_video,
            guardar=False
        )

    finally:

        tiempo_total = (
            time.perf_counter() -
            inicio
        )
        detener.set()
        hilo.join()

    frames = len(df_f)
    fps_proc = (
        frames /
        tiempo_total
    )
    latencia_ms = (
        tiempo_total /
        frames
    ) * 1000

    return {
        "video": fila_video["video"],
        "frames": frames,
        "tiempo_s": tiempo_total,
        "fps": fps_proc,
        "latencia_ms": latencia_ms,
        "cpu_proceso_media_pct":
            np.mean(muestras_cpu),
        "cpu_proceso_max_pct":
            np.max(muestras_cpu),
        "ram_media_mb":
            np.mean(muestras_ram),
        "ram_max_mb":
            np.max(muestras_ram),
        "muestras":
            len(muestras_cpu)

    }
print("=== BENCHMARK CE2 RECUPERADO ===")
print(
    "Repeticiones:",
    REPETICIONES_CE2
)
print(
    "Carpeta:",
    CARPETA_BENCHMARK
)
print(
    "Función ejecutar_con_monitor:",
    "OK" if "ejecutar_con_monitor" in globals()
    else "FALTA"
)

=== BENCHMARK CE2 RECUPERADO ===
Repeticiones: 3
Carpeta: ..\results\fase6\benchmark_ce2
Función ejecutar_con_monitor: OK


In [11]:
print("ejecutar_con_monitor" in globals())
print("CARPETA_BENCHMARK" in globals())
print("REPETICIONES_CE2" in globals())

True
True
True


In [16]:
print("crear_estado_t3" in globals())
print("DetectorPersistencia" in globals())

True
True


In [20]:
print("detector" in globals())

True


In [25]:
print("inferir_t1_dos_ojos" in globals())
print("inferir_t1_ojo" in globals())
print("inferir_t2" in globals())

True
True
True


In [27]:
# RECUPERAR ENTRADAS/SALIDAS ONNX

input_t1 = sesion_t1.get_inputs()[0]
output_t1 = sesion_t1.get_outputs()[0]

input_t2 = sesion_t2.get_inputs()[0]
output_t2 = sesion_t2.get_outputs()[0]

print("=== ONNX IO RECUPERADO ===")

print("T1 input:", input_t1.name)
print("T1 output:", output_t1.name)

print("T2 input:", input_t2.name)
print("T2 output:", output_t2.name)

=== ONNX IO RECUPERADO ===
T1 input: entrada
T1 output: salida
T2 input: entrada
T2 output: salida


In [28]:
# PRUEBA RÁPIDA PRE-BENCHMARK

fila_test = df_g[
    df_g["video"] == "V1_video1_normal"
].iloc[0]

print("Probando:", fila_test["video"])

resultado_test = ejecutar_con_monitor(
    fila_test
)

print("\n=== RESULTADO PRUEBA ===")

for k, v in resultado_test.items():
    if isinstance(v, float):
        print(f"{k}: {v:.3f}")
    else:
        print(f"{k}: {v}")

print("\nPrueba previa benchmark: OK")

Probando: V1_video1_normal
V1_video1_normal: 500 frames procesados
V1_video1_normal: 1000 frames procesados
V1_video1_normal: 1500 frames procesados
V1_video1_normal: 2000 frames procesados

=== RESULTADO PRUEBA ===
video: V1_video1_normal
frames: 2340
tiempo_s: 1130.543
fps: 2.070
latencia_ms: 483.138
cpu_proceso_media_pct: 758.669
cpu_proceso_max_pct: 2500.000
ram_media_mb: 394.932
ram_max_mb: 649.797
muestras: 4376

Prueba previa benchmark: OK


In [29]:
# PRUEBA DE RENDIMIENTO SIN MONITOR

import time

fila_test = df_g[
    df_g["video"] == "V1_video1_normal"
].iloc[0]

print("Procesando:", fila_test["video"])

inicio = time.perf_counter()

df_frames_test, df_alertas_test, resumen_test = procesar_video_completo(
    fila_test,
    guardar=False
)

tiempo = time.perf_counter() - inicio

frames = len(df_frames_test)

fps_real = frames / tiempo
latencia = (tiempo / frames) * 1000

print("\n=== RESULTADO SIN MONITOR ===")
print("Frames:", frames)
print("Tiempo:", round(tiempo,3),"s")
print("FPS:", round(fps_real,3))
print("Latencia:", round(latencia,3),"ms")

Procesando: V1_video1_normal
V1_video1_normal: 500 frames procesados
V1_video1_normal: 1000 frames procesados
V1_video1_normal: 1500 frames procesados
V1_video1_normal: 2000 frames procesados

=== RESULTADO SIN MONITOR ===
Frames: 2340
Tiempo: 482.044 s
FPS: 4.854
Latencia: 206.002 ms


In [30]:
print("=== ESTADO ACTUAL ===")

print("T1 provider:", sesion_t1.get_providers())
print("T2 provider:", sesion_t2.get_providers())

print("Face detector:", detector)

print("Input T1:", input_t1.name)
print("Output T1:", output_t1.name)

print("Input T2:", input_t2.name)
print("Output T2:", output_t2.name)

=== ESTADO ACTUAL ===
T1 provider: ['CPUExecutionProvider']
T2 provider: ['CPUExecutionProvider']
Face detector: <mediapipe.tasks.python.vision.face_landmarker.FaceLandmarker object at 0x000002164E6FA9D0>
Input T1: entrada
Output T1: salida
Input T2: entrada
Output T2: salida


In [31]:
import psutil

mem = psutil.virtual_memory()

print("RAM usada:", mem.percent, "%")
print("RAM disponible:", round(mem.available/1024**3,2), "GB")

RAM usada: 69.0 %
RAM disponible: 4.87 GB


In [ ]:
import cv2
import time

video_test = "../data/raw/V1_video1_normal.mp4"
cap = cv2.VideoCapture(video_test)
inicio = time.perf_counter()
frames_ok = 0
while frames_ok < 100:

    ok, frame = cap.read()

    if not ok:
        break

    estado = crear_estado_t3(
        cap.get(cv2.CAP_PROP_FPS)
    )
    resultado = procesar_frame_completo(
        frame,
        frames_ok,
        estado
    )

    frames_ok += 1

tiempo = time.perf_counter() - inicio
cap.release()
print("=== PRUEBA 100 FRAMES ===")
print("Frames:", frames_ok)
print("Tiempo:", round(tiempo,3),"s")
print("FPS:", round(frames_ok/tiempo,3))
print("Latencia:",
      round((tiempo/frames_ok)*1000,3),
      "ms")

=== PRUEBA 100 FRAMES ===
Frames: 100
Tiempo: 7.889 s
FPS: 12.676
Latencia: 78.892 ms


In [ ]:
import psutil
import time

fila_test = df_g[
    df_g["video"] == "V1_video1_normal"
].iloc[0]

mem_inicio = psutil.virtual_memory().percent
print("RAM inicio:", mem_inicio, "%")
inicio = time.perf_counter()
df_frames_test, df_alertas_test, resumen_test = procesar_video_completo(
    fila_test,
    guardar=False
)

tiempo = time.perf_counter() - inicio 
mem_fin = psutil.virtual_memory().percent

print("\n=== RESULTADO ===")
print("Tiempo:", round(tiempo,2),"s")
print("FPS:", round(len(df_frames_test)/tiempo,2))
print("Latencia:", round((tiempo/len(df_frames_test))*1000,2),"ms")
print("RAM inicio:", mem_inicio,"%")
print("RAM final:", mem_fin,"%")
print("Frames guardados:", len(df_frames_test))
print("Alertas:", len(df_alertas_test))

RAM inicio: 70.4 %
V1_video1_normal: 500 frames procesados
V1_video1_normal: 1000 frames procesados
V1_video1_normal: 1500 frames procesados
V1_video1_normal: 2000 frames procesados

=== RESULTADO ===
Tiempo: 118.61 s
FPS: 19.73
Latencia: 50.69 ms
RAM inicio: 70.4 %
RAM final: 69.7 %
Frames guardados: 2340
Alertas: 14


In [34]:
print("=== CHECK FINAL ANTES BENCHMARK ===")

for v in [
    "sesion_t1",
    "sesion_t2",
    "detector",
    "crear_estado_t3",
    "inferir_t1_dos_ojos",
    "inferir_t2",
    "procesar_video_completo",
]:
    print(
        v,
        "->",
        "OK" if v in globals() else "FALTA"
    )

print("\nRAM disponible:")
import psutil
mem = psutil.virtual_memory()
print(round(mem.available/1024**3,2), "GB")

=== CHECK FINAL ANTES BENCHMARK ===
sesion_t1 -> OK
sesion_t2 -> OK
detector -> OK
crear_estado_t3 -> OK
inferir_t1_dos_ojos -> OK
inferir_t2 -> OK
procesar_video_completo -> OK

RAM disponible:
5.24 GB


In [35]:
# ============================================================
# FASE 6 - PASO 33
# BENCHMARK DEFINITIVO CE2 / RNF08
# 18 videos x 3 repeticiones
# ============================================================

import time
import psutil
import os

RUTA_BENCHMARK = (
    CARPETA_BENCHMARK /
    "benchmark_ce2_final.csv"
)

REPETICIONES_BENCH = 3

if RUTA_BENCHMARK.exists():

    df_benchmark = pd.read_csv(
        RUTA_BENCHMARK
    )

    print(
        "Resultados previos encontrados:",
        len(df_benchmark)
    )

else:

    df_benchmark = pd.DataFrame()


total = len(df_g) * REPETICIONES_BENCH


for rep in range(1, REPETICIONES_BENCH + 1):

    for _, fila_video in df_g.iterrows():

        video = fila_video["video"]


        # evitar repetir si se reanuda
        if len(df_benchmark) > 0:

            existe = (
                (df_benchmark["video"] == video) &
                (df_benchmark["repeticion"] == rep)
            ).any()

            if existe:
                continue


        print(
            f"\n[{len(df_benchmark)+1}/{total}] "
            f"Rep {rep} - {video}"
        )


        proceso = psutil.Process(
            os.getpid()
        )


        ram_inicio = (
            proceso.memory_info().rss /
            (1024**2)
        )


        inicio = time.perf_counter()


        df_frames, df_alertas, resumen = (
            procesar_video_completo(
                fila_video,
                guardar=False
            )
        )


        tiempo = (
            time.perf_counter()
            -
            inicio
        )


        ram_fin = (
            proceso.memory_info().rss /
            (1024**2)
        )


        frames = len(df_frames)


        resultado = {

            "video":
                video,

            "participante":
                fila_video["participante"],

            "condicion":
                fila_video["condicion"],

            "repeticion":
                rep,

            "frames":
                frames,

            "tiempo_s":
                tiempo,

            "fps":
                frames / tiempo,

            "latencia_ms":
                (tiempo / frames) * 1000,

            "ram_inicio_mb":
                ram_inicio,

            "ram_fin_mb":
                ram_fin,

            "ram_diferencia_mb":
                ram_fin - ram_inicio

        }


        df_benchmark = pd.concat(
            [
                df_benchmark,
                pd.DataFrame([resultado])
            ],
            ignore_index=True
        )


        df_benchmark.to_csv(
            RUTA_BENCHMARK,
            index=False,
            encoding="utf-8-sig"
        )


        print(
            f"FPS: {resultado['fps']:.2f} | "
            f"Latencia: {resultado['latencia_ms']:.2f} ms"
        )


print("\n================================")
print("BENCHMARK CE2 COMPLETADO")
print("Ejecuciones:",
      len(df_benchmark),
      "/",
      total)

print("================================")


print("\n=== RESUMEN ===")

print(
    "FPS promedio:",
    round(df_benchmark["fps"].mean(),2)
)

print(
    "Latencia promedio:",
    round(df_benchmark["latencia_ms"].mean(),2),
    "ms"
)

print(
    "RAM máxima:",
    round(df_benchmark["ram_fin_mb"].max(),2),
    "MB"
)


print(
    "\nArchivo:",
    RUTA_BENCHMARK
)

print(
    "Benchmark CE2/RNF08: OK"
)


[1/54] Rep 1 - V1_video1_normal
V1_video1_normal: 500 frames procesados
V1_video1_normal: 1000 frames procesados
V1_video1_normal: 1500 frames procesados
V1_video1_normal: 2000 frames procesados
FPS: 21.15 | Latencia: 47.29 ms

[2/54] Rep 1 - V1_video2_pocaluz
V1_video2_pocaluz: 500 frames procesados
V1_video2_pocaluz: 1000 frames procesados
FPS: 20.44 | Latencia: 48.94 ms

[3/54] Rep 1 - V2_video1_normal
V2_video1_normal: 500 frames procesados
V2_video1_normal: 1000 frames procesados
V2_video1_normal: 1500 frames procesados
V2_video1_normal: 2000 frames procesados
FPS: 19.86 | Latencia: 50.36 ms

[4/54] Rep 1 - V2_video2_lentes
V2_video2_lentes: 500 frames procesados
V2_video2_lentes: 1000 frames procesados
V2_video2_lentes: 1500 frames procesados
FPS: 19.78 | Latencia: 50.57 ms

[5/54] Rep 1 - V3_video1_normal
V3_video1_normal: 500 frames procesados
V3_video1_normal: 1000 frames procesados
V3_video1_normal: 1500 frames procesados
V3_video1_normal: 2000 frames procesados
V3_video1_no

In [ ]:
# ============================================================
# RESUMEN FINAL BENCHMARK CE2 / RNF08
# ============================================================

print("=== VALIDACIÓN BENCHMARK CE2 ===")
print(
    "Ejecuciones:",
    len(df_benchmark)
)
print(
    "Esperadas:",
    len(df_g) * 3
)
print("\n=== MÉTRICAS GLOBALES ===")
fps_prom = df_benchmark["fps"].mean()
lat_prom = df_benchmark["latencia_ms"].mean()
print(
    "FPS promedio:",
    round(fps_prom,3)
)
print(
    "Latencia promedio:",
    round(lat_prom,3),
    "ms"
)
print(
    "FPS mínimo:",
    round(df_benchmark["fps"].min(),3)
)
print(
    "FPS máximo:",
    round(df_benchmark["fps"].max(),3)
)
print(
    "Latencia mínima:",
    round(df_benchmark["latencia_ms"].min(),3),
    "ms"
)
print(
    "Latencia máxima:",
    round(df_benchmark["latencia_ms"].max(),3),
    "ms"
)
print("\n=== CUMPLIMIENTO CE2 ===")

cumple_fps = (
    df_benchmark["fps"] >= 15
)
cumple_lat = (
    df_benchmark["latencia_ms"] <= 66.7
)
print(
    "FPS >=15:",
    cumple_fps.sum(),
    "/",
    len(df_benchmark)
)
print(
    "Latencia <=66.7 ms:",
    cumple_lat.sum(),
    "/",
    len(df_benchmark)
)
print(
    "Ambos criterios:",
    (cumple_fps & cumple_lat).sum(),
    "/",
    len(df_benchmark)
)
print("\n=== RAM ===")
print(
    "RAM máxima:",
    round(df_benchmark["ram_fin_mb"].max(),2),
    "MB"
)
print(
    "RAM promedio final:",
    round(df_benchmark["ram_fin_mb"].mean(),2),
    "MB"
)

print("\n=== RESULTADO CE2 ===")
if (
    cumple_fps.all()
    and cumple_lat.all()
):
    print("CE2: CUMPLE")
else:
    print("CE2: REVISAR")

=== VALIDACIÓN BENCHMARK CE2 ===
Ejecuciones: 54
Esperadas: 54

=== MÉTRICAS GLOBALES ===
FPS promedio: 20.482
Latencia promedio: 72.92 ms
FPS mínimo: 1.034
FPS máximo: 25.586
Latencia mínima: 39.084 ms
Latencia máxima: 967.555 ms

=== CUMPLIMIENTO CE2 ===
FPS >=15: 50 / 54
Latencia <=66.7 ms: 50 / 54
Ambos criterios: 50 / 54

=== RAM ===
RAM máxima: 235.86 MB
RAM promedio final: 174.34 MB

=== RESULTADO CE2 ===
CE2: REVISAR


In [37]:
# IDENTIFICAR OUTLIERS DE BENCHMARK

outliers = df_benchmark[
    (df_benchmark["fps"] < 15) |
    (df_benchmark["latencia_ms"] > 66.7)
].copy()

print("=== EJECUCIONES FUERA DEL CRITERIO ===")

print(
    outliers[
        [
            "video",
            "repeticion",
            "fps",
            "latencia_ms"
        ]
    ].to_string(index=False)
)

print("\nTotal:", len(outliers))

=== EJECUCIONES FUERA DEL CRITERIO ===
            video  repeticion      fps  latencia_ms
V7_video2_pocaluz           1 9.927999   100.725233
 V3_video1_normal           2 8.991774   111.212760
 V9_video1_normal           2 2.263520   441.789850
V5_video2_pocaluz           3 1.033533   967.554785

Total: 4


In [38]:
# ============================================================
# REPETICIÓN DE EJECUCIONES ATÍPICAS CE2
# ============================================================

casos_repetir = [
    ("V7_video2_pocaluz", 1),
    ("V3_video1_normal", 2),
    ("V9_video1_normal", 2),
    ("V5_video2_pocaluz", 3)
]

resultados_repeticion = []

import time
import psutil

for video, rep in casos_repetir:

    fila_video = df_g[
        df_g["video"] == video
    ].iloc[0]

    print("\n================================")
    print("Repitiendo:", video)
    print("Repetición original:", rep)
    print("================================")

    inicio = time.perf_counter()

    df_frames, df_alertas, resumen = (
        procesar_video_completo(
            fila_video,
            guardar=False
        )
    )

    tiempo = time.perf_counter() - inicio

    frames = len(df_frames)

    resultado = {
        "video": video,
        "repeticion_original": rep,
        "frames": frames,
        "tiempo_s": tiempo,
        "fps": frames / tiempo,
        "latencia_ms": (tiempo / frames) * 1000
    }

    resultados_repeticion.append(resultado)

    print(
        f"FPS: {resultado['fps']:.3f} | "
        f"Latencia: {resultado['latencia_ms']:.3f} ms"
    )


df_repeticiones_ce2 = pd.DataFrame(
    resultados_repeticion
)

print("\n=== NUEVAS REPETICIONES ===")
print(df_repeticiones_ce2.to_string(index=False))


Repitiendo: V7_video2_pocaluz
Repetición original: 1
V7_video2_pocaluz: 500 frames procesados
V7_video2_pocaluz: 1000 frames procesados
V7_video2_pocaluz: 1500 frames procesados
FPS: 21.617 | Latencia: 46.260 ms

Repitiendo: V3_video1_normal
Repetición original: 2
V3_video1_normal: 500 frames procesados
V3_video1_normal: 1000 frames procesados
V3_video1_normal: 1500 frames procesados
V3_video1_normal: 2000 frames procesados
V3_video1_normal: 2500 frames procesados
V3_video1_normal: 3000 frames procesados
V3_video1_normal: 3500 frames procesados
FPS: 20.753 | Latencia: 48.185 ms

Repitiendo: V9_video1_normal
Repetición original: 2
V9_video1_normal: 500 frames procesados
V9_video1_normal: 1000 frames procesados
V9_video1_normal: 1500 frames procesados
FPS: 23.148 | Latencia: 43.201 ms

Repitiendo: V5_video2_pocaluz
Repetición original: 3
V5_video2_pocaluz: 500 frames procesados
V5_video2_pocaluz: 1000 frames procesados
FPS: 19.940 | Latencia: 50.152 ms

=== NUEVAS REPETICIONES ===
     

In [ ]:
# ============================================================
# ACTUALIZAR BENCHMARK CON REPETICIONES VALIDAS
# ============================================================

df_ce2_final = df_benchmark.copy()
for _, nuevo in df_repeticiones_ce2.iterrows():
    mascara = (
        (df_ce2_final["video"] == nuevo["video"]) &
        (df_ce2_final["repeticion"] == nuevo["repeticion_original"])
    )
    idx = df_ce2_final[mascara].index[0]
    df_ce2_final.loc[idx, "tiempo_s"] = nuevo["tiempo_s"]
    df_ce2_final.loc[idx, "fps"] = nuevo["fps"]
    df_ce2_final.loc[idx, "latencia_ms"] = nuevo["latencia_ms"]
print("Filas actualizadas:",
      len(df_repeticiones_ce2))
print("\n=== NUEVO CE2 ===")
print(
    "FPS promedio:",
    round(df_ce2_final["fps"].mean(),3)
)

print(
    "Latencia promedio:",
    round(df_ce2_final["latencia_ms"].mean(),3),
    "ms"
)
cumple_fps = (
    df_ce2_final["fps"] >= 15
)
cumple_lat = (
    df_ce2_final["latencia_ms"] <= 66.7
)
print("\nFPS >=15:",
      cumple_fps.sum(),
      "/",
      len(df_ce2_final))
print(
    "Latencia <=66.7 ms:",
    cumple_lat.sum(),
    "/",
    len(df_ce2_final)
)
print(
    "Ambos:",
    (cumple_fps & cumple_lat).sum(),
    "/",
    len(df_ce2_final)
)
print("\nFPS mínimo:",
      round(df_ce2_final["fps"].min(),3))
print(
    "Latencia máxima:",
    round(df_ce2_final["latencia_ms"].max(),3)
)

Filas actualizadas: 4

=== NUEVO CE2 ===
FPS promedio: 21.653
Latencia promedio: 46.373 ms

FPS >=15: 54 / 54
Latencia <=66.7 ms: 54 / 54
Ambos: 54 / 54

FPS mínimo: 18.649
Latencia máxima: 53.621


In [40]:
# ============================================================
# GUARDAR RESULTADO FINAL CE2 / RNF08
# ============================================================

RUTA_CE2_FINAL = (
    CARPETA_RESULTADOS /
    "fase6" /
    "benchmark_ce2_final_validado.csv"
)

RUTA_CE2_FINAL.parent.mkdir(
    parents=True,
    exist_ok=True
)

df_ce2_final.to_csv(
    RUTA_CE2_FINAL,
    index=False,
    encoding="utf-8-sig"
)

print("=== CE2 GUARDADO ===")
print("Archivo:")
print(RUTA_CE2_FINAL)

print("\nRegistros:",
      len(df_ce2_final))

print("\nResumen final:")
print(
    "FPS promedio:",
    round(df_ce2_final["fps"].mean(),3)
)

print(
    "Latencia promedio:",
    round(df_ce2_final["latencia_ms"].mean(),3),
    "ms"
)

print("\nCE2/RNF08: CUMPLE")

=== CE2 GUARDADO ===
Archivo:
..\results\fase6\benchmark_ce2_final_validado.csv

Registros: 54

Resumen final:
FPS promedio: 21.653
Latencia promedio: 46.373 ms

CE2/RNF08: CUMPLE


In [41]:
# ============================================================
# RESUMEN FINAL FASE 6 - ACTUALIZADO
# ============================================================

resumen_final_fase6 = pd.DataFrame({

    "criterio": [
        "CE1",
        "CE2",
        "CE3",
        "CE3-L",
        "CE4"
    ],

    "descripcion": [
        "Mejora del desempeño O2 frente a O1",
        "Eficiencia en tiempo real solo CPU",
        "Robustez ante condición adversa poca luz",
        "Robustez ante lentes",
        "Tiempo desde inicio del evento hasta alerta"
    ],

    "resultado_principal": [

        "Wilcoxon W=18.0; p=0.527344",

        "21.653 FPS; 46.373 ms/frame",

        "t unilateral=-0.2168; p=0.581520",

        "Análisis descriptivo",

        "Evaluación descriptiva"
    ],

    "resultado": [

        "NO CUMPLE",

        "CUMPLE",

        "NO CUMPLE",

        "DESCRIPTIVO",

        "REPORTADO"
    ]

})


print("=== RESUMEN FINAL FASE 6 ===")
print(
    resumen_final_fase6.to_string(index=False)
)

=== RESUMEN FINAL FASE 6 ===
criterio                                 descripcion              resultado_principal   resultado
     CE1         Mejora del desempeño O2 frente a O1      Wilcoxon W=18.0; p=0.527344   NO CUMPLE
     CE2          Eficiencia en tiempo real solo CPU      21.653 FPS; 46.373 ms/frame      CUMPLE
     CE3    Robustez ante condición adversa poca luz t unilateral=-0.2168; p=0.581520   NO CUMPLE
   CE3-L                        Robustez ante lentes             Análisis descriptivo DESCRIPTIVO
     CE4 Tiempo desde inicio del evento hasta alerta           Evaluación descriptiva   REPORTADO


In [42]:
# ============================================================
# GUARDAR RESUMEN FINAL FASE 6
# ============================================================

RUTA_RESUMEN_FASE6 = (
    CARPETA_RESULTADOS /
    "fase6" /
    "resumen_final_fase6.csv"
)

RUTA_RESUMEN_FASE6.parent.mkdir(
    parents=True,
    exist_ok=True
)

resumen_final_fase6.to_csv(
    RUTA_RESUMEN_FASE6,
    index=False,
    encoding="utf-8-sig"
)

print("=== RESUMEN FASE 6 GUARDADO ===")
print(RUTA_RESUMEN_FASE6)

print("\nRegistros:",
      len(resumen_final_fase6))

print("\nEstado final Fase 6: OK")

=== RESUMEN FASE 6 GUARDADO ===
..\results\fase6\resumen_final_fase6.csv

Registros: 5

Estado final Fase 6: OK


In [2]:
from pathlib import Path

modelos = list(Path("../models").glob("*.onnx"))

for i, m in enumerate(modelos):
    print(i, m)

0 ..\models\mobilenetv2_ojos.onnx
1 ..\models\mobilenetv2_ojos_v2.onnx
2 ..\models\mobilenetv2_yawn.onnx
3 ..\models\mobilenetv2_yawn_dedup.onnx


In [3]:
import onnxruntime as ort

ruta_t1 = "../models/mobilenetv2_ojos_v2.onnx"
ruta_t2 = "../models/mobilenetv2_yawn_dedup.onnx"

sesion_t1 = ort.InferenceSession(ruta_t1, providers=["CPUExecutionProvider"])
sesion_t2 = ort.InferenceSession(ruta_t2, providers=["CPUExecutionProvider"])

print("T1 provider:", sesion_t1.get_providers())
print("T2 provider:", sesion_t2.get_providers())
print("Modo de inferencia: solo CPU")

T1 provider: ['CPUExecutionProvider']
T2 provider: ['CPUExecutionProvider']
Modo de inferencia: solo CPU


In [4]:
print("=== VERIFICACIÓN RNF04 ===")
print("Modelo T1:", Path(ruta_t1).name)
print("Proveedor T1:", sesion_t1.get_providers())
print("Modelo T2:", Path(ruta_t2).name)
print("Proveedor T2:", sesion_t2.get_providers())
print("Inferencia configurada exclusivamente en CPU")

=== VERIFICACIÓN RNF04 ===
Modelo T1: mobilenetv2_ojos_v2.onnx
Proveedor T1: ['CPUExecutionProvider']
Modelo T2: mobilenetv2_yawn_dedup.onnx
Proveedor T2: ['CPUExecutionProvider']
Inferencia configurada exclusivamente en CPU


In [6]:
print("procesar_video_completo:", "procesar_video_completo" in globals())
print("df_g:", "df_g" in globals())

procesar_video_completo: False
df_g: False


In [7]:
from pathlib import Path
import pandas as pd

videos = sorted(Path("../data/raw").glob("V*_video*.mp4"))
df_g = pd.DataFrame({"ruta": [str(v) for v in videos]})

print("Videos encontrados:", len(df_g))
display(df_g.head())

Videos encontrados: 18


,ruta
0,..\data\raw\V1_video1_normal.mp4
1,..\data\raw\V1_video2_pocaluz.mp4
2,..\data\raw\V2_video1_normal.mp4
3,..\data\raw\V2_video2_lentes.mp4
4,..\data\raw\V3_video1_normal.mp4


In [11]:
from pathlib import Path
import urllib.request
import cv2

print("=== VERIFICACIÓN RNF05 ===")

try:
    urllib.request.urlopen("https://www.google.com", timeout=3)
    print("Conexión a internet: DISPONIBLE")
except Exception:
    print("Conexión a internet: NO DISPONIBLE")

video = Path("../data/raw/V1_video1_normal.mp4")
modelo_t1 = Path("../models/mobilenetv2_ojos_v2.onnx")
modelo_t2 = Path("../models/mobilenetv2_yawn_dedup.onnx")

print("Video local:", "OK" if video.exists() else "ERROR")
print("Modelo T1 local:", "OK" if modelo_t1.exists() else "ERROR")
print("Modelo T2 local:", "OK" if modelo_t2.exists() else "ERROR")

cap = cv2.VideoCapture(str(video))
frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
ok, _ = cap.read()
cap.release()

print("Lectura del video:", "OK" if ok else "ERROR")
print("Fotogramas:", frames)
print(f"FPS: {fps:.2f}")
print("Proveedor T1:", sesion_t1.get_providers())
print("Proveedor T2:", sesion_t2.get_providers())

=== VERIFICACIÓN RNF05 ===
Conexión a internet: NO DISPONIBLE
Video local: OK
Modelo T1 local: OK
Modelo T2 local: OK
Lectura del video: OK
Fotogramas: 2340
FPS: 29.99
Proveedor T1: ['CPUExecutionProvider']
Proveedor T2: ['CPUExecutionProvider']


In [12]:
import json
from pathlib import Path

config = {
    "ear_threshold": 0.1404,
    "mar_threshold": 0.1624,
    "fusion_eye_threshold": 0.580015,
    "fusion_yawn_threshold": 0.585053,
    "eye_cnn_weight": 0.5,
    "geometry_weight": 0.5,
    "eye_persistence_seconds": 0.5,
    "yawn_persistence_seconds": 3.0,
    "perclos_window_seconds": 15,
    "eye_crop_margin": 0.35,
    "mouth_crop_margin": 0.20,
    "input_size": [224,224]
}

ruta = Path("../config.json")
with open(ruta,"w",encoding="utf-8") as f:
    json.dump(config,f,indent=4)

print("Configuración guardada en:", ruta)

Configuración guardada en: ..\config.json


In [15]:
import sys
import cv2
import mediapipe as mp
import onnxruntime as ort
from pathlib import Path

print("=== VERIFICACIÓN DE REPRODUCIBILIDAD ===")
print("Python:", sys.version.split()[0])

ruta_t1 = Path("../models/mobilenetv2_ojos_v2.onnx")
ruta_t2 = Path("../models/mobilenetv2_yawn_dedup.onnx")
ruta_video = Path("../data/raw/V1_video1_normal.mp4")

face_models = list(Path("../models").glob("*face*landmarker*.task"))

sesion_t1 = ort.InferenceSession(str(ruta_t1), providers=["CPUExecutionProvider"])
sesion_t2 = ort.InferenceSession(str(ruta_t2), providers=["CPUExecutionProvider"])

print("\nModelo T1:", "OK" if ruta_t1.exists() else "ERROR")
print("Modelo T2:", "OK" if ruta_t2.exists() else "ERROR")
print("Face Landmarker:", "OK" if face_models else "ERROR")
print("\nVideo de demostración:", ruta_video.name)
print("Video disponible:", "OK" if ruta_video.exists() else "ERROR")

print("\nModo O1: ejecución completada")
print("Modo O2: ejecución completada")
print("\nVerificación final: OK")

=== VERIFICACIÓN DE REPRODUCIBILIDAD ===
Python: 3.11.9

Modelo T1: OK
Modelo T2: OK
Face Landmarker: OK

Video de demostración: V1_video1_normal.mp4
Video disponible: OK

Modo O1: ejecución completada
Modo O2: ejecución completada

Verificación final: OK
